# Colab 05 (Full): RL (Whole-Episode GRPO + Optional Official Unsloth Step GRPO)
- `RL_jssp_fssp.py` 대응 노트북
- 메인 경로는 `grpo_episode`: step-conditioned env rollout을 끝까지 수행한 뒤 final makespan으로만 GRPO update
- `unsloth_grpo`는 공식 Unsloth step-dataset GRPO 실험 경로이고, `reinforce` / `grpo_manual` / `bopo`는 legacy/manual alias 경로
- random problem 또는 dataset 기반 에피소드 생성 지원
- strict parsing + step masking + optional step reflection 포함



In [ ]:
!pip -q install -U unsloth tqdm
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets accelerate bitsandbytes trl peft


## 1) 설정


In [ ]:
import contextlib
import json

import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
from torch.optim import AdamW
from datasets import load_dataset, Dataset
from huggingface_hub import login, hf_hub_download, HfApi, upload_folder, snapshot_download
from tqdm import trange
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'max_seq_length': 16384,
    'model_type': 'llama8b',
    'model_path': '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128/checkpoint-19000',
    'model_checkpoint_tag': None,
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # RL loop
    'epochs': 1,
    'episodes_per_epoch': 50,
    'rl_log_every_episodes': 5,
    'bopo_update_every_episodes': 5,
    'print_episode_summary': True,
    'learning_rate': 1e-5,
    'temperature': 0.8,
    'top_p': 0.95,
    'top_k': 40,
    'max_new_tokens': 1,  # legacy field; action decoding uses step_action_max_new_tokens

    'baseline_beta': 0.9,
    'use_running_baseline': False,

    # algo
    'rl_algo': 'grpo_episode',  # grpo_episode | unsloth_grpo | bopo | reinforce | grpo_manual
    'group_size': 4,
    'grpo_epochs': 2,
    'grpo_update_batch_size': 1,
    'clip_epsilon': 0.2,
    'kl_coef': 0.0,
    'bopo_beta': 1.5,
    'bopo_gap_scale': 2.0,
    'bopo_margin': 0.0,
    'bopo_min_relative_gap': 0.01,
    'bopo_max_pairs_per_group': 64,
    'bopo_max_step_pairs_per_pair': 8,
    'bopo_pair_mode': 'shared_prefix',
    'reward_mode': 'raw_neg_makespan',
    'rl_update_mode': 'adapter_only',


# official Unsloth GRPO
'grpo_dataset_source': 'hf',  # hf | local
'grpo_dataset_path': None,
'grpo_dataset_role': 'policy',  # policy only for stable action-code GRPO
'grpo_step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
'grpo_step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
'grpo_step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
'grpo_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_policy_step_train_dispatch_v1',
'grpo_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_policy_dispatch.jsonl',
'grpo_step_dataset_local_path_dispatch': '/content/jssp_step_train_policy_dispatch.jsonl',
'grpo_max_dataset_rows': 5000,
'grpo_min_feasible_actions': 2,
'grpo_shuffle_seed': 42,
'grpo_per_device_train_batch_size': 1,
'grpo_gradient_accumulation_steps': 1,
'grpo_num_generations': 6,
'grpo_num_train_epochs': None,
'grpo_max_steps': -1,
'grpo_logging_steps': 5,
'grpo_save_steps': 100,
'grpo_max_prompt_length': 4096,
'grpo_max_completion_length': 8,
'grpo_report_to': 'none',
'grpo_optim': 'adamw_8bit',
'grpo_weight_decay': 0.01,
'grpo_warmup_ratio': 0.05,
'grpo_lr_scheduler_type': 'linear',
'grpo_max_grad_norm': 1.0,
'grpo_use_fast_inference': False,
'grpo_gpu_memory_utilization': 0.6,
'grpo_reward_valid_weight': 1.0,
'grpo_reward_proxy_weight': 1.0,
'grpo_reward_teacher_weight': 0.25,

    # step decoding/prompt
    'step_action_max_new_tokens': 8,
    'disable_step_problem_context': False,
    'enable_step_improvement': False,
    'step_reflection_passes': 1,
    'step_reflection_topk': 3,
    'invalid_makespan_penalty': 1e6,
    'disable_masking': False,
    'action_code_width': 4,
    'action_code_seed': 42,
    'env_mode': 'dispatch',  # serial | dispatch
    'action_code_cap': 9999,
    'print_step_trace': False,

    # data source
    'use_random_problems': False,
    'dataset_source': 'hf',  # hf | local
    'dataset_hf_repo': 'HYUNJINI/jssp_raw_source_v1',
    'dataset_hf_file': 'llm_jssp/train.json',
    'dataset_local_path': '/content/train.json',
    'dataset_path': None,
    'hf_token': HF_TOKEN,

    # random problem settings
    'random_jobs': 10,
    'random_machines': 10,
    'random_time_low': 1,
    'random_time_high': 100,

    # raw RL problem-size filter (HF/local dataset path only)
    'rl_num_jobs': None,
    'rl_num_machines': None,
    'min_rl_num_jobs': 10,
    'min_rl_num_machines': 10,

    'seed': 42,

    # save/upload
    'output_dir': '/content/drive/MyDrive/LLM_JSSP_checkpoints/jssp_mixed_dispatch_llama8b_r128_grpo_episode',
    'save_every_epoch': True,
    'save_every_n_epochs': 1,
    'save_rl_epoch_metrics': True,
    'save_rl_episode_metrics': True,
    'upload_after_train': False,
    'hf_model_repo_out': 'HYUNJINI/jssp_mixed_dispatch_llama8b_r128_grpo_episode_v1',
    'upload_source': 'latest_checkpoint',
    'checkpoint_tag': 'checkpoint-epoch-1',
}

BOPO_PRESETS = {
    'fast_stable': {
        'disable_masking': False,
        'step_action_max_new_tokens': 8,
        'temperature': 0.8,
        'top_p': 0.95,
        'top_k': 40,
        'group_size': 4,
        'bopo_beta': 1.5,
        'bopo_gap_scale': 2.0,
        'bopo_margin': 0.0,
        'bopo_min_relative_gap': 0.01,
        'bopo_max_pairs_per_group': 64,
        'bopo_max_step_pairs_per_pair': 8,
        'enable_step_improvement': False,
        'print_step_trace': False,
    },
    'high_performance': {
        'disable_masking': False,
        'step_action_max_new_tokens': 8,
        'temperature': 0.8,
        'top_p': 0.92,
        'top_k': 40,
        'group_size': 6,
        'bopo_beta': 2.5,
        'bopo_gap_scale': 4.0,
        'bopo_margin': 0.0,
        'bopo_min_relative_gap': 0.01,
        'bopo_max_pairs_per_group': 192,
        'bopo_max_step_pairs_per_pair': 24,
        'enable_step_improvement': False,
        'print_step_trace': False,
    },
}

# choose one: fast_stable | high_performance
CFG['bopo_preset'] = 'fast_stable'
CFG.update(BOPO_PRESETS[CFG['bopo_preset']])

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit',
    'qwen2.5_7b': 'unsloth/Qwen2.5-7B-Instruct-unsloth-bnb-4bit',
    'qwen2.5_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
}

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

print(CFG)


## 2) 유틸/환경/GRPO 함수 (inline)


In [ ]:
import math

import os

import random

import re

import time

from functools import lru_cache

from dataclasses import dataclass

from pathlib import Path

from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np

import torch

import torch.nn.functional as F

from torch.optim import AdamW

from tqdm import tqdm, trange

def parse_matrix_form_jssp(matrix_content: str, sep: str = ' '):
    lines = [ln.strip() for ln in matrix_content.strip().splitlines() if ln.strip()]
    n, m = map(int, lines[0].split(sep))
    inst_for_ortools = []
    for i in range(n):
        nums = list(map(int, lines[i + 1].split(sep)))
        if len(nums) != 2 * m:
            raise ValueError(f'row {i} has {len(nums)} ints, expected {2*m}')
        inst_for_ortools.append([[nums[2 * j], nums[2 * j + 1]] for j in range(m)])
    ms = None
    if len(lines) >= n + 2:
        try:
            ms = float(lines[n + 1])
        except Exception:
            ms = None
    return n, m, inst_for_ortools, ms

def generate_random_instance(num_jobs=10, num_machines=10, process_time_range=(1, 100), rng=None):
    if rng is None:
        rng = np.random.default_rng()

    inst_for_ortools = []
    for _ in range(num_jobs):
        machines = rng.permutation(num_machines).tolist()
        durations = rng.integers(process_time_range[0], process_time_range[1] + 1, size=num_machines).tolist()
        inst_for_ortools.append([[int(m), int(t)] for m, t in zip(machines, durations)])

    return {'inst_for_ortools': inst_for_ortools}

ACTION_TOKEN_PREFIX = "A"

LEGACY_ACTION_TOKEN_PREFIX = "S"

ACTION_CODE_PATTERN = re.compile(
    r"(?:Action\s*:\s*)?<\s*[aAsS]\s*(\d+)\s*>",
    re.IGNORECASE,
)

def format_action_code(
    code_index: int,
    code_width: int = 4,
    prefix: str = ACTION_TOKEN_PREFIX,
) -> str:
    if int(code_index) < 0:
        raise ValueError(f"code_index must be non-negative, got {code_index}.")
    if int(code_width) < 1:
        raise ValueError(f"code_width must be >= 1, got {code_width}.")
    return f"<{str(prefix)}{int(code_index):0{int(code_width)}d}>"

def parse_action_code(text: str, code_width: int = 4) -> Optional[str]:
    if not isinstance(text, str):
        return None
    match = ACTION_CODE_PATTERN.search(text)
    if not match:
        return None
    return format_action_code(int(match.group(1)), code_width=code_width)

class StepActionParseError(ValueError):
    """Raised when a generated step action cannot be parsed."""

def _normalize_text(text: str) -> str:
    return text.replace("\r", "")

def action_code_to_token_id(tokenizer, action_code: str) -> int:
    token_id = tokenizer.convert_tokens_to_ids(str(action_code))
    if token_id is None:
        raise ValueError(f"Tokenizer returned None token id for action_code={action_code!r}")
    token_id = int(token_id)
    unk_id = getattr(tokenizer, 'unk_token_id', None)
    if unk_id is not None and token_id == int(unk_id):
        encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
        if len(encoded) == 1:
            return int(encoded[0])
        raise ValueError(
            f"Action code {action_code!r} is not a single tokenizer token. "
            "Special tokens were likely not installed correctly."
        )
    return token_id

def token_id_to_action_code(tokenizer, token_id: int, code_width: int = 4) -> Optional[str]:
    token = tokenizer.convert_ids_to_tokens(int(token_id))
    if isinstance(token, bytes):
        token = token.decode('utf-8', errors='ignore')
    token_text = str(token) if token is not None else ''
    parsed = parse_action_code(token_text, code_width=code_width)
    if parsed is not None:
        return parsed
    decoded = tokenizer.decode([int(token_id)], skip_special_tokens=False)
    return parse_action_code(decoded, code_width=code_width)

def _is_prefix_of_any(candidate_text: str, feasible_action_codes: Sequence[str]) -> bool:
    candidate_text = _normalize_text(candidate_text)
    for code in feasible_action_codes:
        if code.startswith(candidate_text):
            return True
    return False

def build_step_prefix_allowed_tokens_fn(
    tokenizer,
    feasible_action_codes_provider: Iterable[str],
    prompt_len: int = 0,
    code_width: int = 4,
    code_cap: int = 9999,
):
    feasible_action_codes = tuple(str(code) for code in feasible_action_codes_provider)
    if not feasible_action_codes:
        raise RuntimeError("No feasible action codes available at this step.")

    feasible_token_ids = []
    for action_code in feasible_action_codes:
        token_id = tokenizer.convert_tokens_to_ids(str(action_code))
        if token_id is None:
            raise RuntimeError(f"Tokenizer returned None token id for action_code={action_code!r}")
        token_id = int(token_id)
        unk_id = getattr(tokenizer, 'unk_token_id', None)
        if unk_id is not None and token_id == int(unk_id):
            encoded = tokenizer.encode(str(action_code), add_special_tokens=False)
            if len(encoded) != 1:
                raise RuntimeError(
                    f"Action code {action_code!r} is not a single tokenizer token."
                )
            token_id = int(encoded[0])
        feasible_token_ids.append(token_id)
    feasible_token_ids = tuple(sorted(set(feasible_token_ids)))
    eos_token_id = getattr(tokenizer, "eos_token_id", None)

    def prefix_allowed_tokens_fn(batch_id: int, input_ids) -> List[int]:
        if hasattr(input_ids, "tolist"):
            input_ids = input_ids.tolist()
        generated_ids = input_ids[int(prompt_len) :]
        if len(generated_ids) == 0:
            return list(feasible_token_ids)

        chosen_action_code = token_id_to_action_code(
            tokenizer,
            int(generated_ids[0]),
            code_width=code_width,
        )
        if (
            len(generated_ids) == 1
            and chosen_action_code is not None
            and str(chosen_action_code) in feasible_action_codes
            and eos_token_id is not None
        ):
            return [int(eos_token_id)]

        if len(generated_ids) == 1 and chosen_action_code is not None and str(chosen_action_code) in feasible_action_codes:
            return list(feasible_token_ids)

        generated_text = _normalize_text(
            tokenizer.decode(generated_ids, skip_special_tokens=False)
        )
        raise RuntimeError(
            "No valid next tokens for step action generation. "
            f"generated_text={generated_text!r}, feasible_action_codes={list(feasible_action_codes)}"
        )

    return prefix_allowed_tokens_fn

HEADER_PATTERN = re.compile(
    r"JSSP\s+with\s+(\d+)\s+Jobs,\s*(\d+)\s+Machines",
    re.IGNORECASE,
)

JOB_HEADER_PATTERN = re.compile(
    r"^\s*Job\s+(\d+)\s+consists\s+of\s+Operations:\s*$",
    re.IGNORECASE,
)

OPERATION_PATTERN = re.compile(
    r"^\s*Operation\s+(\d+):\s*M(\d+),\s*(\d+)\s*$",
    re.IGNORECASE,
)

SOLUTION_OPERATION_PATTERN = re.compile(
    r"Job\s*(\d+)\s*Operation\s*(\d+),\s*M(\d+)",
    re.IGNORECASE,
)

MAKESPAN_PATTERN = re.compile(r"Makespan\s*:\s*(\d+)", re.IGNORECASE)

@dataclass(frozen=True)
class ParsedTeacherAction:
    """Parsed action from one-shot output text."""

    job_id: int
    op_idx: int
    machine_id: int

def parse_prompt_jobs_first(prompt_text: str, strict: bool = True) -> List[List[List[int]]]:
    """
    Parse `prompt_jobs_first` text into `inst_for_ortools` format.

    Returns:
        inst_for_ortools[job][op] = [machine, duration]
    """
    if not isinstance(prompt_text, str) or not prompt_text.strip():
        raise ValueError("prompt_text must be a non-empty string.")

    header_match = HEADER_PATTERN.search(prompt_text)
    expected_jobs = int(header_match.group(1)) if header_match else None
    expected_machines = int(header_match.group(2)) if header_match else None

    jobs: Dict[int, List[List[int]]] = {}
    current_job: Optional[int] = None

    for raw_line in prompt_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        job_match = JOB_HEADER_PATTERN.match(line)
        if job_match:
            current_job = int(job_match.group(1))
            jobs.setdefault(current_job, [])
            continue

        op_match = OPERATION_PATTERN.match(line)
        if op_match:
            if current_job is None:
                raise ValueError(f"Operation line appears before job header: {raw_line!r}")

            op_idx = int(op_match.group(1))
            machine = int(op_match.group(2))
            duration = int(op_match.group(3))
            expected_op_idx = len(jobs[current_job])
            if strict and op_idx != expected_op_idx:
                raise ValueError(
                    f"Operation index mismatch in Job {current_job}: "
                    f"expected {expected_op_idx}, got {op_idx}."
                )
            jobs[current_job].append([machine, duration])

    if not jobs:
        raise ValueError("Failed to parse any jobs from prompt_text.")

    max_job_id = max(jobs.keys())
    if strict and set(jobs.keys()) != set(range(max_job_id + 1)):
        raise ValueError("Job IDs must be contiguous from 0 when strict=True.")

    inst_for_ortools = [jobs[job_id] for job_id in range(max_job_id + 1)]

    if strict:
        if expected_jobs is not None and expected_jobs != len(inst_for_ortools):
            raise ValueError(
                f"Header jobs ({expected_jobs}) != parsed jobs ({len(inst_for_ortools)})."
            )
        if expected_machines is not None:
            for job_id, job_ops in enumerate(inst_for_ortools):
                if len(job_ops) != expected_machines:
                    raise ValueError(
                        f"Job {job_id} has {len(job_ops)} ops, expected {expected_machines}."
                    )

    return inst_for_ortools

def parse_solution_actions(
    solution_text: str, strict: bool = True
) -> Tuple[List[ParsedTeacherAction], Optional[int]]:
    """
    Parse one-shot output into per-step teacher actions.

    Returns:
        actions in decode order, declared makespan from text if present.
    """
    if not isinstance(solution_text, str) or not solution_text.strip():
        raise ValueError("solution_text must be a non-empty string.")

    matches = SOLUTION_OPERATION_PATTERN.findall(solution_text)
    if not matches:
        raise ValueError("No 'Job X Operation Y, MZ' actions found in solution_text.")

    next_expected_op: Dict[int, int] = {}
    actions: List[ParsedTeacherAction] = []
    for job_str, op_str, machine_str in matches:
        job_id = int(job_str)
        op_idx = int(op_str)
        machine_id = int(machine_str)

        expected_op_idx = next_expected_op.get(job_id, 0)
        if strict and op_idx != expected_op_idx:
            raise ValueError(
                f"Teacher action order violation for job {job_id}: "
                f"expected op {expected_op_idx}, got {op_idx}."
            )

        next_expected_op[job_id] = op_idx + 1
        actions.append(ParsedTeacherAction(job_id=job_id, op_idx=op_idx, machine_id=machine_id))

    makespan_match = MAKESPAN_PATTERN.search(solution_text)
    declared_makespan = int(makespan_match.group(1)) if makespan_match else None
    return actions, declared_makespan

class StaticJSSPStepEnv:
    """
    Deterministic static JSSP environment for step-by-step action selection.

    Action:
        choose one feasible job id at each step.
    """

    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [
            sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools
        ]
        self.total_ops = sum(self.operations_per_job)

        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(
            machine for job_ops in self.inst_for_ortools for machine, _ in job_ops
        )
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.scheduled_ops = 0
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    @classmethod
    def from_prompt_jobs_first(cls, prompt_text: str, strict: bool = True) -> "StaticJSSPStepEnv":
        inst_for_ortools = parse_prompt_jobs_first(prompt_text, strict=strict)
        return cls(inst_for_ortools)

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.scheduled_ops = 0
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def get_feasible_jobs(self) -> List[int]:
        return [
            job_id
            for job_id in range(self.num_jobs)
            if self.job_next_op[job_id] < self.operations_per_job[job_id]
        ]

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [
            f"M{int(machine_id)}:{int(duration)}"
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]
        ]

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines

        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1

        return machine_remaining_load, machine_remaining_ops

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)

            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)

            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(
                float(total_ops - rem_ops) / float(total_ops)
            )
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = (
            float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        )
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = (
            float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_var = (
            sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time)
            / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = (
            machine_remaining_load.index(bottleneck_machine_load)
            if machine_remaining_load
            else -1
        )
        bottleneck_machine_ops_left = (
            int(machine_remaining_ops[bottleneck_machine_id])
            if bottleneck_machine_id >= 0
            else 0
        )

        state = {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": (
                float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0
            ),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
        }
        return state

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")

        op_idx = self.job_next_op[job_id]
        if op_idx >= self.operations_per_job[job_id]:
            raise ValueError(f"Job {job_id} has no remaining operation.")

        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = max(self.job_ready_time[job_id], self.machine_ready_time[machine_id])
        end_time = start_time + duration

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": job_id,
            "op_idx": op_idx,
            "machine_id": machine_id,
            "duration": duration,
            "start_time": start_time,
            "end_time": end_time,
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

    def rollout_teacher(self, job_sequence: Iterable[int]) -> List[Dict[str, object]]:
        """
        Roll out a full teacher sequence and collect step-level records.
        """
        self.reset()
        records: List[Dict[str, object]] = []
        for step_idx, job_id in enumerate(job_sequence):
            state_before = self.get_state_json()
            feasible_jobs = state_before["feasible_jobs"]
            if job_id not in feasible_jobs:
                raise ValueError(
                    f"Infeasible teacher action at step {step_idx}: "
                    f"job {job_id}, feasible={feasible_jobs}."
                )
            _, _, done, info = self.step(job_id)
            records.append(
                {
                    "step_idx": step_idx,
                    "state_json": state_before,
                    "feasible_jobs": list(feasible_jobs),
                    "target_job": job_id,
                    "info": info,
                    "done": done,
                }
            )
        if not self.is_done():
            raise ValueError(
                f"Teacher sequence ended early: scheduled {self.scheduled_ops}/{self.total_ops}."
            )
        return records

    def get_event_log(self) -> List[Dict[str, int]]:
        return list(self.event_log)

def _format_machine_ready(machine_ready_time: Sequence[int]) -> str:
    return " ".join(f"M{i}={t}" for i, t in enumerate(machine_ready_time))

def _format_feasible_jobs(feasible_jobs: Sequence[int]) -> str:
    if not feasible_jobs:
        return "[]"
    return "[" + ", ".join(f"Job {j}" for j in feasible_jobs) + "]"

def _format_action_codes(action_codes: Sequence[str]) -> str:
    if not action_codes:
        return "[]"
    return "[" + ", ".join(str(code) for code in action_codes) + "]"

def _format_route_tokens(route_tokens: Sequence[str]) -> str:
    if not route_tokens:
        return "[]"
    return "[" + ", ".join(str(token) for token in route_tokens) + "]"

def _safe_ratio(numerator: float, denominator: float) -> float:
    if float(denominator) == 0.0:
        return 0.0
    return float(numerator) / float(denominator)

def _format_value(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value)

def _machine_token(machine_id: int) -> str:
    return f"M{int(machine_id)}" if int(machine_id) >= 0 else "M-"

def build_randomized_action_code_map(
    feasible_jobs: Sequence[int],
    rng: Optional[random.Random] = None,
    code_width: int = 4,
    code_start: int = 1,
    code_cap: int = 9999,
) -> Dict[str, int]:
    """
    Create randomized action-code mapping for one step.

    Example output:
        {"<A0102>": 7, "<A4377>": 3, ...}
    """
    jobs = [int(j) for j in feasible_jobs]
    if len(set(jobs)) != len(jobs):
        raise ValueError(f"feasible_jobs contains duplicates: {jobs}")

    if code_start < 0:
        raise ValueError(f"code_start must be non-negative, got {code_start}")
    if code_cap < code_start:
        raise ValueError(
            f"code_cap must be >= code_start, got code_start={code_start}, code_cap={code_cap}"
        )
    if len(jobs) > (int(code_cap) - int(code_start) + 1):
        raise ValueError(
            "Not enough action-code slots for this step: "
            f"jobs={len(jobs)}, available={int(code_cap) - int(code_start) + 1}, "
            f"range=[{code_start}, {code_cap}]"
        )

    if not jobs:
        return {}

    shuffled_jobs = list(jobs)
    if rng is None:
        random.shuffle(shuffled_jobs)
        sampled_code_indices = random.sample(
            range(int(code_start), int(code_cap) + 1), k=len(shuffled_jobs)
        )
    else:
        rng.shuffle(shuffled_jobs)
        sampled_code_indices = rng.sample(
            range(int(code_start), int(code_cap) + 1), k=len(shuffled_jobs)
        )

    action_code_to_job: Dict[str, int] = {}
    for code_idx, job_id in zip(sampled_code_indices, shuffled_jobs):
        code = format_action_code(int(code_idx), code_width=code_width)
        action_code_to_job[code] = int(job_id)
    return action_code_to_job

def invert_action_code_map(action_code_to_job: Dict[str, int]) -> Dict[int, str]:
    job_to_action_code: Dict[int, str] = {}
    for code, job_id in action_code_to_job.items():
        if job_id in job_to_action_code:
            raise ValueError(f"Duplicate job id in action_code_to_job: {job_id}")
        job_to_action_code[int(job_id)] = str(code)
    return job_to_action_code

def build_problem_context_text(inst_for_ortools: Sequence[Sequence[Sequence[int]]]) -> str:
    """
    Build minimal static problem context text.
    """
    num_jobs = len(inst_for_ortools)
    total_ops = sum(len(job_ops) for job_ops in inst_for_ortools)
    max_machine = -1
    for job_ops in inst_for_ortools:
        for machine_id, _ in job_ops:
            max_machine = max(max_machine, int(machine_id))
    num_machines = max_machine + 1 if max_machine >= 0 else 0

    return f"Problem: {num_jobs} jobs x {num_machines} machines (total_ops={total_ops})"

def summarize_global_dynamic_state(state_json: Dict[str, object]) -> Dict[str, object]:
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(state_json.get("remaining_work", [])))  # type: ignore[arg-type]
    )
    summary = {
        "step_idx": int(state_json["step_idx"]),
        "total_steps": int(state_json["total_steps"]),
        "scheduled_ratio": float(state_json.get("scheduled_ratio", 0.0)),
        "current_cmax": current_cmax,
        "total_remaining_work": total_remaining_work,
        "unfinished_jobs_count": int(state_json.get("unfinished_jobs_count", 0)),
        "unfinished_jobs_ratio": float(state_json.get("unfinished_jobs_ratio", 0.0)),
        "machine_ready_min": int(state_json.get("machine_ready_min", 0)),
        "machine_ready_mean": float(state_json.get("machine_ready_mean", 0.0)),
        "machine_ready_max": int(state_json.get("machine_ready_max", 0)),
        "machine_ready_std": float(state_json.get("machine_ready_std", 0.0)),
        "bottleneck_machine_id": int(state_json.get("bottleneck_machine_id", -1)),
        "bottleneck_machine_load": int(state_json.get("bottleneck_machine_load", 0)),
        "bottleneck_machine_ops_left": int(state_json.get("bottleneck_machine_ops_left", 0)),
    }
    return summary

def compute_action_transition_features(
    state_json: Dict[str, object],
    action_code_to_job: Dict[str, int],
) -> Tuple[int, List[Dict[str, object]]]:
    job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
    next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
    next2_machine: List[int] = state_json.get("next2_machine", [-1] * len(next_machine))  # type: ignore[assignment]
    next2_proc_time: List[int] = state_json.get("next2_proc_time", [0] * len(next_machine))  # type: ignore[assignment]
    remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
    remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
    job_total_ops: List[int] = state_json.get("job_total_ops", [1] * len(next_machine))  # type: ignore[assignment]
    job_total_work: List[int] = state_json.get("job_total_work", [1] * len(next_machine))  # type: ignore[assignment]
    machine_remaining_load: List[int] = state_json.get("machine_remaining_load", [0] * len(machine_ready_time))  # type: ignore[assignment]
    machine_remaining_ops: List[int] = state_json.get("machine_remaining_ops", [0] * len(machine_ready_time))  # type: ignore[assignment]
    post_route_tokens_by_job: List[List[str]] = state_json.get("post_route_tokens", [[] for _ in next_machine])  # type: ignore[assignment]

    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(int(x) for x in remaining_work))
    )
    current_time = int(state_json.get("current_time", 0))

    effects: List[Dict[str, object]] = []
    for action_code, job_id in action_code_to_job.items():
        job = int(job_id)
        machine_id = int(next_machine[job])
        proc_time = int(next_proc_time[job])
        if machine_id < 0 or proc_time <= 0:
            continue

        job_ready_before = int(job_ready_time[job])
        machine_ready_before = int(machine_ready_time[machine_id])
        est_start = max(current_time, job_ready_before, machine_ready_before)
        est_end = est_start + proc_time
        current_cmax_after = max(current_cmax, est_end)
        delta_cmax = current_cmax_after - current_cmax
        rem_ops_before = int(remaining_ops[job])
        rem_ops_after = max(0, rem_ops_before - 1)
        rem_work_before = int(remaining_work[job])
        rem_work_after = max(0, rem_work_before - proc_time)
        total_ops = max(int(job_total_ops[job]), 1)
        total_work = max(int(job_total_work[job]), 1)
        job_progress_ratio_before = float(total_ops - rem_ops_before) / float(total_ops)
        job_progress_ratio_after = float(total_ops - rem_ops_after) / float(total_ops)
        affected_machine_load = int(machine_remaining_load[machine_id])
        affected_machine_ops_left = int(machine_remaining_ops[machine_id])

        effect = {
            "action_code": str(action_code),
            "job_id": job,
            "machine_id": machine_id,
            "machine_token": _machine_token(machine_id),
            "proc_time": proc_time,
            "next_machine": machine_id,
            "next_proc_time": proc_time,
            "next2_machine": int(next2_machine[job]),
            "next2_proc_time": int(next2_proc_time[job]),
            "remaining_ops_before": rem_ops_before,
            "remaining_ops_after": rem_ops_after,
            "remaining_work_before": rem_work_before,
            "remaining_work_after": rem_work_after,
            "job_progress_ratio_before": float(job_progress_ratio_before),
            "job_progress_ratio_after": float(job_progress_ratio_after),
            "job_ready_before": job_ready_before,
            "job_ready_after": int(est_end),
            "machine_ready_before": machine_ready_before,
            "machine_ready_after": int(est_end),
            "estimated_start": int(est_start),
            "estimated_end": int(est_end),
            "est_start": int(est_start),
            "est_end": int(est_end),
            "current_cmax_before": int(current_cmax),
            "current_cmax_after": int(current_cmax_after),
            "estimated_makespan_after": int(current_cmax_after),
            "delta_cmax": int(delta_cmax),
            "delta_cmax_ratio": float(_safe_ratio(delta_cmax, max(current_cmax_after, 1))),
            "job_wait": int(max(0, machine_ready_before - job_ready_before)),
            "machine_idle_gap": int(max(0, job_ready_before - machine_ready_before)),
            "slack_to_current_cmax": int(current_cmax - est_end),
            "affected_machine_load": affected_machine_load,
            "affected_machine_ops_left": affected_machine_ops_left,
            "affected_machine_load_ratio": float(
                _safe_ratio(affected_machine_load, max(total_remaining_work, 1))
            ),
            "remaining_work_after_ratio": float(_safe_ratio(rem_work_after, total_work)),
            "post_route_tokens": list(post_route_tokens_by_job[job]),
            "post_route_len": int(len(post_route_tokens_by_job[job])),
        }
        effects.append(effect)

    effects.sort(
        key=lambda x: (
            int(x["estimated_makespan_after"]),
            int(x["estimated_start"]),
            int(x["proc_time"]),
            int(x["job_id"]),
        )
    )
    return int(current_cmax), effects

def render_action_transition_line(effect: Dict[str, object]) -> str:
    return (
        f"{effect['action_code']} | "
        f"operation machine={_machine_token(int(effect['next_machine']))} | "
        f"processing time={effect['next_proc_time']} | "
        f"rem_ops:{effect['remaining_ops_before']}->{effect['remaining_ops_after']} | "
        f"rem_work:{effect['remaining_work_before']}->{effect['remaining_work_after']} | "
        f"job_prog:{_format_value(effect['job_progress_ratio_before'])}->{_format_value(effect['job_progress_ratio_after'])} | "
        f"job_ready:{effect['job_ready_before']}->{effect['job_ready_after']} | "
        f"machine_ready:{effect['machine_ready_before']}->{effect['machine_ready_after']} | "
        f"est_start={effect['estimated_start']} | "
        f"est_end={effect['estimated_end']} | "
        f"cmax:{effect['current_cmax_before']}->{effect['current_cmax_after']} | "
        f"delta_cmax={effect['delta_cmax']} | "
        f"delta_cmax_ratio={_format_value(effect['delta_cmax_ratio'])} | "
        f"job_wait={effect['job_wait']} | "
        f"machine_idle_gap={effect['machine_idle_gap']} | "
        f"slack_to_cmax={effect['slack_to_current_cmax']} | "
        f"machine_load={effect['affected_machine_load']} | "
        f"machine_ops_left={effect['affected_machine_ops_left']} | "
        f"machine_load_ratio={_format_value(effect['affected_machine_load_ratio'])} | "
        f"rem_work_after_ratio={_format_value(effect['remaining_work_after_ratio'])} | "
        f"post_route={_format_route_tokens(effect.get('post_route_tokens', []))}"
    )

def build_step_prompt(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    step_idx: int,
    total_steps: int,
    problem_context_text: Optional[str] = None,
    action_code_to_job: Optional[Dict[str, int]] = None,
) -> str:
    """
    Build deterministic compact prompt for one-step action selection.

    Output format is always one line.
    - Default (legacy): Action: Job <id>
    - Action-token mode: <Axxxx>
    """
    lines = [
        "You are solving JSSP step-by-step.",
        "Objective: Minimize final makespan (Cmax) while keeping every action feasible.",
    ]
    if problem_context_text:
        lines.extend(
            [
                "Static problem context:",
                problem_context_text,
            ]
        )

    global_summary = summarize_global_dynamic_state(state_json)
    lines.extend(
        [
            "Global dynamic state:",
            (
                f"step={int(global_summary['step_idx']) + 1}/{int(global_summary['total_steps'])} "
                f"scheduled_ratio={_format_value(global_summary['scheduled_ratio'])}"
            ),
            f"current_cmax={global_summary['current_cmax']}",
            f"total_remaining_work={global_summary['total_remaining_work']}",
            (
                f"unfinished_jobs_count={global_summary['unfinished_jobs_count']} "
                f"unfinished_jobs_ratio={_format_value(global_summary['unfinished_jobs_ratio'])}"
            ),
            (
                f"machine_ready_min={global_summary['machine_ready_min']} "
                f"machine_ready_mean={_format_value(global_summary['machine_ready_mean'])} "
                f"machine_ready_max={global_summary['machine_ready_max']} "
                f"machine_ready_std={_format_value(global_summary['machine_ready_std'])}"
            ),
            (
                f"bottleneck_machine={_machine_token(int(global_summary['bottleneck_machine_id']))} "
                f"bottleneck_load={global_summary['bottleneck_machine_load']} "
                f"bottleneck_ops_left={global_summary['bottleneck_machine_ops_left']}"
            ),
        ]
    )

    if action_code_to_job:
        action_codes = list(action_code_to_job.keys())
        _, action_effects = compute_action_transition_features(
            state_json=state_json,
            action_code_to_job=action_code_to_job,
        )
        lines.extend(
            [
                f"Feasible action codes: {_format_action_codes(action_codes)}",
                "Candidate transition features:",
            ]
        )
        for effect in action_effects:
            lines.append(render_action_transition_line(effect))
        lines.extend(
            [
                "Action codes are randomized at each step. Do not assume persistent code identity.",
                "Choose exactly one feasible action code.",
                "Return exactly one code from the feasible action set, for example <A3812>.",
            ]
        )
    else:
        job_next_op: List[int] = state_json["job_next_op"]  # type: ignore[assignment]
        next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
        next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
        job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
        remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
        remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
        machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
        lines.extend(
            [
                f"Feasible jobs: {_format_feasible_jobs(feasible_jobs)}",
                "Job state summary:",
            ]
        )
        for job_id in range(len(job_next_op)):
            lines.append(
                (
                    f"Job {job_id}: "
                    f"next_op={job_next_op[job_id]}, "
                    f"next_machine={next_machine[job_id]}, "
                    f"next_proc={next_proc_time[job_id]}, "
                    f"job_ready={job_ready_time[job_id]}, "
                    f"remaining_ops={remaining_ops[job_id]}, "
                    f"remaining_work={remaining_work[job_id]}"
                )
            )
        lines.extend(
            [
                f"Machine ready times: {_format_machine_ready(machine_ready_time)}",
                "Choose exactly one job from feasible jobs to minimize final makespan (Cmax).",
                "Return exactly one line: Action: Job <id>",
            ]
        )

    return "\n".join(lines)

def build_step_improvement_prompt(
    state_text: str,
    candidate_action_text: str,
    feasible_jobs: Sequence[object],
    reflection_memory: Optional[str] = None,
    step_diagnostics: Optional[str] = None,
) -> str:
    """Build a hindsight-aware step-improvement prompt."""
    feasible_str: str
    if feasible_jobs and isinstance(feasible_jobs[0], str):
        feasible_str = _format_action_codes([str(x) for x in feasible_jobs])
        output_format = "<Axxxx>"
    else:
        feasible_str = _format_feasible_jobs([int(x) for x in feasible_jobs])
        output_format = "Action: Job <id>"
    prompt = (
        "You are revising one decision inside a completed JSSP schedule.\n"
        "Use hindsight from the full episode, not only local greedy signals.\n"
        "If a small short-term sacrifice helps earlier bottleneck activation, lower idle loss, "
        "or better downstream route progression, prefer it.\n\n"
        "Current step state:\n"
        f"{state_text}\n\n"
        "Previously chosen action in the failed/improvable rollout:\n"
        f"{candidate_action_text}\n\n"
        "Rules:\n"
        "- Objective: minimize final makespan (Cmax).\n"
        "- Think in hindsight: ask which choice would have reduced downstream bottleneck delay, idle gaps, or route blocking.\n"
        "- Prefer the action that best aligns with bottleneck-machine usage, downstream route progression, and lower regret against strong alternatives.\n"
        f"- Final action must be one of feasible options: {feasible_str}\n"
        f"- Return exactly one output in this format: {output_format}\n"
        "- Do not output explanation.\n"
        "- Do not repeat the previous action if the reflection evidence says an alternative is structurally better.\n"
    )
    if reflection_memory:
        prompt += f"\nEpisode hindsight reflection:\n{reflection_memory}\n"
    if step_diagnostics:
        prompt += f"\nCritical-step diagnostics:\n{step_diagnostics}\n"
    return prompt

def build_step_rationale_prompt(
    state_text: str,
    chosen_job: Optional[int] = None,
    feasible_jobs: Optional[Sequence[int]] = None,
    chosen_action_code: Optional[str] = None,
    feasible_action_codes: Optional[Sequence[str]] = None,
) -> str:
    """
    Build explanation prompt after action selection.

    This prompt is intentionally separated from action decoding so that
    feasibility/format masking of action is unaffected.
    """
    using_codes = chosen_action_code is not None
    if using_codes:
        chosen_label = str(chosen_action_code)
        if feasible_action_codes is None:
            raise ValueError("feasible_action_codes is required when chosen_action_code is used.")
        other_labels = [str(code) for code in feasible_action_codes if str(code) != chosen_label]
        others = ", ".join(other_labels) if other_labels else "(none)"
    else:
        if chosen_job is None or feasible_jobs is None:
            raise ValueError("chosen_job and feasible_jobs are required in legacy rationale mode.")
        chosen_label = f"Job {int(chosen_job)}"
        other_jobs = [int(j) for j in feasible_jobs if int(j) != int(chosen_job)]
        others = ", ".join(f"Job {j}" for j in other_jobs) if other_jobs else "(none)"

    return (
        "Explain why the already-selected action is reasonable for this JSSP step.\n"
        "Focus on final makespan (Cmax) and feasibility.\n\n"
        f"{state_text}\n\n"
        f"Selected action (fixed, do not change): {chosen_label}\n"
        f"Other feasible options: {others}\n\n"
        "Output format:\n"
        "Reason: <one concise sentence>\n"
        "Rules:\n"
        "- Output exactly one line starting with 'Reason:'.\n"
        "- Keep it concise (<= 25 words).\n"
        "- Do not change the selected action.\n"
        "- Do not output any 'Action:' line.\n"
        "- Do not output 'Not chosen:'.\n"
    )

@dataclass
class EpisodeBatch:
    log_probs: torch.Tensor
    advantages: torch.Tensor

@dataclass
class TrajectorySample:
    sequence_ids: torch.Tensor
    prompt_len: int
    action_token_id: int
    reward: float
    advantage: float
    old_log_prob: torch.Tensor
    feasible: bool
    makespan: float

@dataclass
class BOPOStepPair:
    winner_sequence_ids: torch.Tensor
    winner_prompt_len: int
    winner_action_token_id: int
    loser_sequence_ids: torch.Tensor
    loser_prompt_len: int
    loser_action_token_id: int
    relative_gap: float
    winner_makespan: float
    loser_makespan: float

@dataclass
class StepActionTrace:
    sequence_ids: torch.Tensor
    prompt_len: int
    action_token_id: int
    chosen_job: int
    step_idx: int

class ExponentialBaseline:
    """Simple running baseline for REINFORCE style updates."""

    def __init__(self, beta: float = 0.9):
        self.beta = beta
        self.value: Optional[float] = None

    def update(self, reward: float) -> float:
        if self.value is None:
            self.value = reward
        else:
            self.value = self.beta * self.value + (1 - self.beta) * reward
        return self.value

def mwkr_schedule(inst_for_ortools: List[List[List[int]]]) -> Tuple[List[Dict], float]:
    """Most Work Remaining heuristic schedule."""

    num_jobs = len(inst_for_ortools)
    num_machines = len(inst_for_ortools[0])

    job_next = [0] * num_jobs
    job_time = [0.0] * num_jobs
    machine_time = [0.0] * num_machines
    total_remaining = [
        sum(op[1] for op in job_ops) for job_ops in inst_for_ortools
    ]

    schedule = []

    while any(idx < len(inst_for_ortools[j]) for j, idx in enumerate(job_next)):
        # Jobs ready to schedule are those with remaining operations
        ready_jobs = [
            j for j in range(num_jobs) if job_next[j] < len(inst_for_ortools[j])
        ]
        if not ready_jobs:
            break

        # Choose job with maximum remaining work (MWKR)
        job = max(ready_jobs, key=lambda j: total_remaining[j])
        op_idx = job_next[job]
        machine, duration = inst_for_ortools[job][op_idx]

        start = max(job_time[job], machine_time[machine])
        end = start + duration

        schedule.append(
            {
                "Job": job,
                "Operation": op_idx,
                "Machine": machine,
                "Start Time": start,
                "Duration": duration,
                "End Time": end,
            }
        )

        job_next[job] += 1
        job_time[job] = end
        machine_time[machine] = end
        total_remaining[job] -= duration

    makespan = max(job_time) if schedule else float("inf")
    return schedule, makespan

def _count_total_ops(inst_for_ortools: List[List[List[int]]]) -> int:
    return int(sum(len(job_ops) for job_ops in inst_for_ortools))


def compute_episode_reward(
    makespan: float,
    feasible: bool,
    inst_for_ortools: List[List[List[int]]],
    invalid_makespan_penalty: float,
    reward_mode: str = "raw_neg_makespan",
    heuristic_makespan: float | None = None,
) -> float:
    if not feasible or not math.isfinite(float(makespan)):
        if reward_mode == "neg_makespan_per_op":
            total_ops = max(_count_total_ops(inst_for_ortools), 1)
            return -float(invalid_makespan_penalty) / float(total_ops)
        if reward_mode == "mwkr_relative":
            return -1.0
        return -float(invalid_makespan_penalty)

    makespan = float(makespan)
    if reward_mode == "raw_neg_makespan":
        return -makespan
    if reward_mode == "neg_makespan_per_op":
        total_ops = max(_count_total_ops(inst_for_ortools), 1)
        return -(makespan / float(total_ops))
    if reward_mode == "mwkr_relative":
        baseline_ms = float(heuristic_makespan) if heuristic_makespan is not None else float(mwkr_schedule(inst_for_ortools)[1])
        if not math.isfinite(baseline_ms) or baseline_ms <= 0:
            return -makespan
        return (baseline_ms - makespan) / baseline_ms
    raise ValueError(f"Unsupported reward_mode={reward_mode}")

def _build_step_chat_prompt(tokenizer, state_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert JSSP scheduler. "
                "Primary objective: minimize final makespan (Cmax). "
                "Choose exactly one feasible action code for this step. "
                "Output exactly one code such as <A3812> and nothing else."
            ),
        },
        {"role": "user", "content": state_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def _build_step_improvement_chat_prompt(tokenizer, improvement_prompt_text: str) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are refining one JSSP step action. "
                "Primary objective: minimize final makespan (Cmax). "
                "Return exactly one feasible action code such as <A3812>."
            ),
        },
        {"role": "user", "content": improvement_prompt_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def generate_step_action(
    model,
    tokenizer,
    prompt_text: str,
    feasible_action_codes: List[str],
    device: torch.device,
    max_new_tokens: int,
    temperature: float = 1.0,
    top_p: float = 0.95,
    top_k: int = 50,
    use_masking: bool = True,
    action_code_width: int = 4,
    action_code_cap: int = 9999,
) -> Tuple[str, torch.Tensor, int, str]:
    if not feasible_action_codes:
        raise StepActionParseError("No feasible action codes available at this step.")

    prompt_inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    prompt_len = int(prompt_inputs["input_ids"].size(1))

    was_training = model.training
    model.eval()
    try:
        generation_kwargs = dict(
            **prompt_inputs,
            max_new_tokens=1 if use_masking else int(max_new_tokens),
            do_sample=True,
            temperature=float(temperature),
            top_p=float(top_p),
            top_k=int(top_k),
            return_dict_in_generate=False,
        )

        if use_masking:
            generation_kwargs["prefix_allowed_tokens_fn"] = build_step_prefix_allowed_tokens_fn(
                tokenizer=tokenizer,
                feasible_action_codes_provider=list(feasible_action_codes),
                prompt_len=prompt_len,
                code_width=action_code_width,
                code_cap=action_code_cap,
            )

        with torch.no_grad():
            generated = model.generate(**generation_kwargs)

        sequence_ids = generated[0].detach().cpu()
        new_ids = sequence_ids[prompt_len:]
        if new_ids.numel() <= 0:
            raise StepActionParseError("No action token was generated.")

        chosen_action_code = token_id_to_action_code(
            tokenizer,
            int(new_ids[0].item()),
            code_width=action_code_width,
        )
        if chosen_action_code is None:
            generated_text = tokenizer.decode(new_ids, skip_special_tokens=False).strip()
            raise StepActionParseError(
                f"Failed to map generated token to action token. output={generated_text!r}"
            )
        if str(chosen_action_code) not in feasible_action_codes:
            raise StepActionParseError(
                f"Generated action token is not feasible. parsed={chosen_action_code}, feasible={list(feasible_action_codes)}"
            )
        return str(chosen_action_code), sequence_ids, prompt_len, str(chosen_action_code)
    finally:
        if was_training:
            model.train()

def _estimate_action_effects(state_json, action_code_to_job):
    return compute_action_transition_features(state_json, action_code_to_job)

class DispatchJSSPStepEnv:
    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools]
        self.total_ops = sum(self.operations_per_job)
        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(machine for job_ops in self.inst_for_ortools for machine, _ in job_ops)
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.current_time = 0
        self.scheduled_ops = 0
        self.running_ops: List[Dict[str, int]] = []
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.current_time = 0
        self.scheduled_ops = 0
        self.running_ops = []
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [f"M{int(machine_id)}:{int(duration)}" for machine_id, duration in self.inst_for_ortools[job_id][next_op:]]

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines
        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1
        return machine_remaining_load, machine_remaining_ops

    def _dispatchable(self, job_id: int) -> bool:
        if self.job_next_op[job_id] >= self.operations_per_job[job_id]:
            return False
        machine_id, duration = self._next_operation(job_id, offset=0)
        if machine_id < 0 or duration <= 0:
            return False
        return int(self.job_ready_time[job_id]) <= int(self.current_time) and int(self.machine_ready_time[machine_id]) <= int(self.current_time)

    def get_feasible_jobs(self) -> List[int]:
        return [job_id for job_id in range(self.num_jobs) if self._dispatchable(job_id)]

    def _advance_to_next_event(self) -> None:
        if not self.running_ops:
            return
        next_time = min(int(op["end_time"]) for op in self.running_ops)
        self.current_time = int(next_time)
        self.running_ops = [dict(op) for op in self.running_ops if int(op["end_time"]) > int(self.current_time)]

    def _advance_until_decision_epoch(self) -> None:
        while not self.is_done() and not self.get_feasible_jobs():
            if not self.running_ops:
                raise RuntimeError("Dispatch environment deadlocked: no dispatchable jobs and no running ops.")
            self._advance_to_next_event()

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)
            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)
            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(float(total_ops - rem_ops) / float(total_ops))
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time)) if self.machine_ready_time else 0.0
        machine_ready_var = sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time) / float(len(self.machine_ready_time)) if self.machine_ready_time else 0.0
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = machine_remaining_load.index(bottleneck_machine_load) if machine_remaining_load else -1
        bottleneck_machine_ops_left = int(machine_remaining_ops[bottleneck_machine_id]) if bottleneck_machine_id >= 0 else 0

        return {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0,
            "current_time": int(self.current_time),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
            "dispatchable_jobs": self.get_feasible_jobs(),
            "idle_machines": [int(machine_id) for machine_id, ready in enumerate(self.machine_ready_time) if int(ready) <= int(self.current_time)],
            "running_operations": [dict(op) for op in sorted(self.running_ops, key=lambda x: (int(x["end_time"]), int(x["machine_id"]), int(x["job_id"])))],
        }

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")
        feasible_jobs = self.get_feasible_jobs()
        if job_id not in feasible_jobs:
            raise ValueError(f"Job {job_id} is not dispatchable at t={self.current_time}. feasible={feasible_jobs}")

        op_idx = self.job_next_op[job_id]
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = int(self.current_time)
        end_time = start_time + int(duration)

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1
        self.running_ops.append({
            "job_id": int(job_id), "op_idx": int(op_idx), "machine_id": int(machine_id),
            "start_time": int(start_time), "end_time": int(end_time), "duration": int(duration),
        })

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": int(job_id),
            "op_idx": int(op_idx),
            "machine_id": int(machine_id),
            "duration": int(duration),
            "start_time": int(start_time),
            "end_time": int(end_time),
            "decision_time": int(start_time),
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        if not done:
            self._advance_until_decision_epoch()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

def summarize_dispatch_state(state_json: Dict[str, object]) -> Dict[str, object]:
    machine_ready_time: List[int] = state_json["machine_ready_time"]
    current_cmax = int(state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0))
    return {
        "step_idx": int(state_json["step_idx"]),
        "total_steps": int(state_json["total_steps"]),
        "scheduled_ratio": float(state_json.get("scheduled_ratio", 0.0)),
        "current_time": int(state_json.get("current_time", 0)),
        "current_cmax": current_cmax,
        "unfinished_jobs_count": int(state_json.get("unfinished_jobs_count", 0)),
        "unfinished_jobs_ratio": float(state_json.get("unfinished_jobs_ratio", 0.0)),
        "machine_ready_min": int(state_json.get("machine_ready_min", 0)),
        "machine_ready_mean": float(state_json.get("machine_ready_mean", 0.0)),
        "machine_ready_max": int(state_json.get("machine_ready_max", 0)),
        "machine_ready_std": float(state_json.get("machine_ready_std", 0.0)),
        "bottleneck_machine_id": int(state_json.get("bottleneck_machine_id", -1)),
        "bottleneck_machine_load": int(state_json.get("bottleneck_machine_load", 0)),
        "bottleneck_machine_ops_left": int(state_json.get("bottleneck_machine_ops_left", 0)),
    }

def _order_prompt_effects_randomly(state_json: Dict[str, object], action_effects: Sequence[Dict[str, object]]) -> List[Dict[str, object]]:
    ordered_effects = [dict(effect) for effect in action_effects]
    seed_material = f"{int(state_json.get('step_idx', 0))}|{int(state_json.get('current_time', 0))}|{int(state_json.get('current_cmax', 0))}|" + "|".join(sorted(str(effect.get('action_code', '')) for effect in ordered_effects))
    rng = random.Random(seed_material)
    rng.shuffle(ordered_effects)
    return ordered_effects

def build_dispatch_step_prompt(state_json: Dict[str, object], feasible_jobs: Sequence[int], step_idx: int, total_steps: int, problem_context_text: Optional[str] = None, action_code_to_job: Optional[Dict[str, int]] = None) -> str:
    lines = [
        "You are solving JSSP with event-driven dispatching.",
        "Objective: minimize final makespan (Cmax) while respecting precedence and machine availability.",
    ]
    if problem_context_text:
        lines.extend(["Static problem context:", problem_context_text])

    summary = summarize_dispatch_state(state_json)
    idle_machines = [int(x) for x in state_json.get("idle_machines", [])]
    running_operations = list(state_json.get("running_operations", []) or [])
    lines.extend([
        "Dispatch state:",
        f"decision_step={int(summary['step_idx']) + 1}/{int(summary['total_steps'])} scheduled_ratio={_format_value(summary['scheduled_ratio'])}",
        f"current_time={summary['current_time']}",
        f"current_cmax={summary['current_cmax']}",
        f"unfinished_jobs_count={summary['unfinished_jobs_count']} unfinished_jobs_ratio={_format_value(summary['unfinished_jobs_ratio'])}",
        f"idle_machines={[ _machine_token(m) for m in idle_machines ]} num_running_ops={len(running_operations)}",
        f"machine_ready_min={summary['machine_ready_min']} machine_ready_mean={_format_value(summary['machine_ready_mean'])} machine_ready_max={summary['machine_ready_max']} machine_ready_std={_format_value(summary['machine_ready_std'])}",
        f"bottleneck_machine={_machine_token(int(summary['bottleneck_machine_id']))} bottleneck_load={summary['bottleneck_machine_load']} bottleneck_ops_left={summary['bottleneck_machine_ops_left']}",
    ])
    if running_operations:
        lines.append('Running operations:')
        for op in running_operations:
            lines.append(f"Job {int(op['job_id'])} Op {int(op['op_idx'])} on {_machine_token(int(op['machine_id']))}: {int(op['start_time'])}->{int(op['end_time'])}")

    if action_code_to_job:
        action_codes = list(action_code_to_job.keys())
        _, action_effects = compute_action_transition_features(state_json=state_json, action_code_to_job=action_code_to_job)
        lines.extend([
            f"Dispatchable action codes now: {_format_action_codes(action_codes)}",
            'Candidate dispatch effects:',
        ])
        for effect in _order_prompt_effects_randomly(state_json, action_effects):
            lines.append(render_action_transition_line(effect))
        lines.extend([
            'Action codes are randomized at each decision epoch. Do not assume persistent code identity.',
            'Choose exactly one dispatchable action code.',
            'Return exactly one code from the dispatchable action set, for example <A3812>.',
        ])
    else:
        lines.extend([
            f"Dispatchable jobs: {list(int(j) for j in feasible_jobs)}",
            'Choose exactly one dispatchable job.',
            'Return exactly one line: Action: Job <id>',
        ])
    return "\n".join(lines)

def _normalize_env_mode(env_mode: str) -> str:
    resolved = str(env_mode).lower()
    if resolved not in {"serial", "dispatch"}:
        raise ValueError(f"Unsupported env_mode={env_mode}")
    return resolved

def _make_step_env(inst_for_ortools, env_mode: str):
    if _normalize_env_mode(env_mode) == "dispatch":
        return DispatchJSSPStepEnv(inst_for_ortools)
    return StaticJSSPStepEnv(inst_for_ortools)

def _estimate_action_effects(state_json, action_code_to_job, env_mode: str):
    return compute_action_transition_features(state_json, action_code_to_job)

def _build_state_text(state_json, feasible_jobs, step_idx, total_steps, problem_context_text, action_code_to_job, env_mode: str):
    if _normalize_env_mode(env_mode) == "dispatch":
        return build_dispatch_step_prompt(
            state_json=state_json,
            feasible_jobs=feasible_jobs,
            step_idx=step_idx,
            total_steps=total_steps,
            problem_context_text=problem_context_text,
            action_code_to_job=action_code_to_job,
        )
    return build_step_prompt(
        state_json=state_json,
        feasible_jobs=feasible_jobs,
        step_idx=step_idx,
        total_steps=total_steps,
        problem_context_text=problem_context_text,
        action_code_to_job=action_code_to_job,
    )

def _best_alternative_option(step):
    chosen_code = str(step.get("chosen_action_code"))
    alternatives = [
        opt
        for opt in step.get("all_options", [])
        if str(opt.get("action_code")) != chosen_code
    ]
    if not alternatives:
        return None
    return min(
        alternatives,
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("estimated_start", 10**12)),
            int(x.get("proc_time", 10**12)),
        ),
    )

def _critical_step_score(step):
    best_alt = _best_alternative_option(step)
    if best_alt is None:
        return None
    chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
    best_alt_ms = int(best_alt.get("estimated_makespan_after", chosen_ms))
    immediate_gap = int(chosen_ms - best_alt_ms)
    makespan_jump = int(step.get("makespan_after", 0) - step.get("makespan_before", 0))
    chosen_start = int(step.get("chosen_start_time", 0))
    best_alt_start = int(best_alt.get("estimated_start", chosen_start))
    start_delay = int(chosen_start - best_alt_start)
    if immediate_gap <= 0 and makespan_jump <= 0 and start_delay <= 0:
        return None
    return (immediate_gap, makespan_jump, start_delay, int(step.get("step_idx", -1)))

def _select_top_critical_steps(step_records, top_k: int = 3):
    scored = []
    for step in step_records:
        score = _critical_step_score(step)
        if score is None:
            continue
        scored.append((score, step))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [step for _, step in scored[: max(1, int(top_k))]]

def _select_critical_step(step_records):
    top = _select_top_critical_steps(step_records, top_k=1)
    return top[0] if top else None

def _build_step_diagnostics(step):
    chosen_code = str(step.get("chosen_action_code"))
    chosen_job = int(step.get("chosen_job", -1))
    chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
    chosen_start = int(step.get("chosen_start_time", 0))
    alternatives = [
        opt
        for opt in step.get("all_options", [])
        if str(opt.get("action_code")) != chosen_code
    ]
    alternatives = sorted(
        alternatives,
        key=lambda x: (
            int(x.get("estimated_makespan_after", 10**12)),
            int(x.get("estimated_start", 10**12)),
        ),
    )
    lines = [
        (
            f"Chosen={chosen_code}/Job{chosen_job}, est_Cmax_after={chosen_ms}, "
            f"start={chosen_start}, machine=M{int(step.get('machine_id', -1))}"
        )
    ]
    for rank, opt in enumerate(alternatives[:3], start=1):
        lines.append(
            (
                f"Alt{rank}={opt['action_code']}/Job{int(opt['job_id'])}, "
                f"est_Cmax_after={int(opt['estimated_makespan_after'])}, "
                f"est_start={int(opt['estimated_start'])}, "
                f"proc={int(opt['proc_time'])}"
            )
        )
    return "\n".join(lines)

def _build_reflection_memory(current_makespan: float, critical_steps):
    lines = [
        f"Current episode makespan={float(current_makespan):.1f}.",
        "Reflection rules (avoid bottleneck choices, prefer lower estimated Cmax/start):",
    ]
    for rank, step in enumerate(critical_steps, start=1):
        best_alt = _best_alternative_option(step)
        if best_alt is None:
            continue
        chosen_code = str(step.get("chosen_action_code"))
        chosen_job = int(step.get("chosen_job", -1))
        chosen_ms = int(step.get("chosen_estimated_makespan_after", step.get("makespan_after", 0)))
        chosen_start = int(step.get("chosen_start_time", 0))
        alt_code = str(best_alt.get("action_code"))
        alt_job = int(best_alt.get("job_id", -1))
        alt_ms = int(best_alt.get("estimated_makespan_after", chosen_ms))
        alt_start = int(best_alt.get("estimated_start", chosen_start))
        lines.append(
            (
                f"Rule {rank}: step {int(step.get('step_idx', -1))}, avoid {chosen_code}/Job{chosen_job} "
                f"(est_Cmax={chosen_ms}, start={chosen_start}); "
                f"prefer {alt_code}/Job{alt_job} (est_Cmax={alt_ms}, start={alt_start})."
            )
        )
    return "\n".join(lines)

def _print_step_trace(step_records):
    for step in step_records:
        print(
            f"[step {int(step['step_idx']):03d}] ms {int(step['makespan_before'])}->{int(step['makespan_after'])} "
            f"chosen={step['chosen_action_code']}->Job{step['chosen_job']} "
            f"(M{step['machine_id']},t={step['chosen_proc_time']},"
            f"start={step['chosen_start_time']},end={step['chosen_end_time']})"
        )
        not_chosen = step.get("not_chosen_options", [])
        if not_chosen:
            parts = []
            for opt in not_chosen:
                parts.append(
                    f"{opt['action_code']}->Job{opt['job_id']}"
                    f"(M{opt['machine_id']},t={opt['proc_time']},ms={opt['estimated_makespan_after']},"
                    f"dC={opt['delta_cmax']})"
                )
            print("  not_chosen:", "; ".join(parts))

def rollout_step_episode(model, tokenizer, inst_for_ortools: List[List[List[int]]], device: torch.device, step_action_max_new_tokens, env_mode: Optional[str] = None, temperature: float = 1.0, top_p: float = 0.95, top_k: int = 50, use_masking: bool = True, include_problem_context: bool = True, enable_step_improvement: bool = False, step_reflection_passes: int = 1, step_reflection_topk: int = 3, action_code_width: int = 4, action_code_seed: int = 42, action_code_cap: int = 9999, print_step_trace: bool = False) -> Tuple[float, bool, List[StepActionTrace]]:
    if isinstance(step_action_max_new_tokens, dict):
        cfg = dict(step_action_max_new_tokens)
        step_action_max_new_tokens = int(cfg['step_action_max_new_tokens'])
        env_mode = str(cfg.get('env_mode', 'serial'))
        temperature = float(cfg.get('temperature', temperature))
        top_p = float(cfg.get('top_p', top_p))
        top_k = int(cfg.get('top_k', top_k))
        use_masking = not bool(cfg.get('disable_masking', False))
        include_problem_context = not bool(cfg.get('disable_step_problem_context', False))
        enable_step_improvement = bool(cfg.get('enable_step_improvement', enable_step_improvement))
        step_reflection_passes = int(cfg.get('step_reflection_passes', step_reflection_passes))
        step_reflection_topk = int(cfg.get('step_reflection_topk', step_reflection_topk))
        action_code_width = int(cfg.get('action_code_width', action_code_width))
        action_code_seed = int(cfg.get('action_code_seed', action_code_seed))
        action_code_cap = int(cfg.get('action_code_cap', action_code_cap))
        print_step_trace = bool(cfg.get('print_step_trace', print_step_trace))
    elif env_mode is None:
        raise ValueError('env_mode must be provided when rollout_step_episode is called without CFG dict.')

    problem_context_text = build_problem_context_text(inst_for_ortools) if include_problem_context else None

    def _rollout_once(code_seed: int, guidance_by_step=None, reflection_memory_text: str | None = None):
        env = _make_step_env(inst_for_ortools, env_mode)
        env.reset()
        traces: List[StepActionTrace] = []
        step_records = []
        action_rng = random.Random(int(code_seed))
        guidance_map = guidance_by_step or {}

        try:
            while not env.is_done():
                state = env.get_state_json()
                feasible_jobs = [int(j) for j in state['feasible_jobs']]
                step_idx = int(state['step_idx'])
                action_code_to_job = build_randomized_action_code_map(
                    feasible_jobs=feasible_jobs, rng=action_rng, code_width=action_code_width, code_start=1, code_cap=action_code_cap
                )
                makespan_before, action_effects = _estimate_action_effects(state_json=state, action_code_to_job=action_code_to_job, env_mode=env_mode)
                effect_by_code = {x['action_code']: x for x in action_effects}
                feasible_action_codes = list(action_code_to_job.keys())
                state_text = _build_state_text(
                    state_json=state, feasible_jobs=feasible_jobs, step_idx=step_idx, total_steps=int(state['total_steps']),
                    problem_context_text=problem_context_text, action_code_to_job=action_code_to_job, env_mode=env_mode,
                )
                if reflection_memory_text:
                    state_text = f"{state_text}\nEpisode reflection memory:\n{reflection_memory_text}\nApply these reflection rules while selecting this step action."
                if step_idx in guidance_map:
                    step_guidance = dict(guidance_map[step_idx])
                    preferred_job = int(step_guidance.get('preferred_job', -1))
                    avoid_jobs = [int(x) for x in step_guidance.get('avoid_jobs', []) if int(x) >= 0]
                    reason_text = str(step_guidance.get('reason', '')).strip()
                    guide_lines = ['Post-episode guidance: This step was identified as a Cmax bottleneck.']
                    if preferred_job >= 0:
                        guide_lines.append(f'If feasible, prefer Job {preferred_job}.')
                    if avoid_jobs:
                        guide_lines.append('Avoid these jobs if strong alternatives exist: ' + ', '.join(f'Job {j}' for j in avoid_jobs))
                    if reason_text:
                        guide_lines.append(f'Why: {reason_text}')
                    state_text = f"{state_text}\n" + '\n'.join(guide_lines)

                base_prompt = _build_step_chat_prompt(tokenizer, state_text)
                chosen_action_code, sequence_ids, prompt_len, _ = generate_step_action(
                    model=model, tokenizer=tokenizer, prompt_text=base_prompt, feasible_action_codes=feasible_action_codes, device=device,
                    max_new_tokens=step_action_max_new_tokens, temperature=temperature, top_p=top_p, top_k=top_k, use_masking=use_masking,
                    action_code_width=action_code_width, action_code_cap=action_code_cap,
                )
                chosen_job = int(action_code_to_job[chosen_action_code])
                chosen_effect = effect_by_code.get(chosen_action_code)
                if chosen_effect is None:
                    raise RuntimeError('Internal mismatch: chosen action code is missing in step option effects. ' + f'chosen_action_code={chosen_action_code}, available={list(effect_by_code.keys())}')

                _, _, _, info = env.step(chosen_job)
                traces.append(StepActionTrace(sequence_ids=sequence_ids, prompt_len=prompt_len, action_token_id=int(sequence_ids[prompt_len].item()), chosen_job=chosen_job, step_idx=step_idx))
                step_records.append({
                    'step_idx': int(step_idx), 'state_text': state_text, 'feasible_action_codes': feasible_action_codes, 'action_code_to_job': action_code_to_job,
                    'chosen_action_code': str(chosen_action_code), 'chosen_job': int(chosen_job), 'machine_id': int(info['machine_id']),
                    'chosen_start_time': int(info['start_time']), 'chosen_end_time': int(info['end_time']), 'chosen_proc_time': int(info['duration']),
                    'makespan_before': int(makespan_before), 'makespan_after': int(info['makespan_so_far']),
                    'chosen_estimated_makespan_after': int(chosen_effect['estimated_makespan_after']), 'all_options': action_effects,
                    'not_chosen_options': [x for x in action_effects if str(x['action_code']) != str(chosen_action_code)], 'guidance_applied': bool(step_idx in guidance_map),
                })
        except StepActionParseError:
            raise
        except Exception as exc:
            raise RuntimeError(f'Unexpected rollout error at step {step_idx}: {exc}') from exc

        return float(env.get_makespan()), True, traces, step_records, []

    makespan, feasible, traces, step_records, notes = _rollout_once(code_seed=int(action_code_seed), guidance_by_step=None, reflection_memory_text=None)
    if not feasible:
        return makespan, feasible, traces

    best_makespan = float(makespan)
    best_traces = traces
    best_step_records = step_records
    improvement_notes = list(notes)

    if enable_step_improvement and int(step_reflection_passes) > 0:
        if print_step_trace:
            print(f"[Episode Improvement] start: passes={int(step_reflection_passes)}, topk={max(1, int(step_reflection_topk))}, baseline_makespan={float(best_makespan):.1f}")

        reflection_memory_text = ''
        for pass_idx in range(int(step_reflection_passes)):
            critical_steps = _select_top_critical_steps(best_step_records, top_k=max(1, int(step_reflection_topk)))
            if not critical_steps:
                if print_step_trace:
                    print(f'[Episode Improvement] pass {pass_idx + 1}: no critical steps found.')
                break

            reflection_memory_text = _build_reflection_memory(best_makespan, critical_steps)
            guidance_by_step = {}
            for step in critical_steps:
                best_alt = _best_alternative_option(step)
                if best_alt is None:
                    continue
                chosen_job = int(step.get('chosen_job', -1))
                alt_job = int(best_alt.get('job_id', -1))
                guidance_by_step[int(step['step_idx'])] = {
                    'preferred_job': alt_job,
                    'avoid_jobs': [chosen_job] if chosen_job >= 0 else [],
                    'reason': _build_step_diagnostics(step),
                }

            reroll_code_seed = int(action_code_seed) + 1000 + pass_idx
            try:
                candidate_makespan, candidate_feasible, candidate_traces, candidate_records, candidate_notes = _rollout_once(code_seed=reroll_code_seed, guidance_by_step=guidance_by_step, reflection_memory_text=reflection_memory_text)
            except StepActionParseError as exc:
                improvement_notes.append(f'pass={pass_idx + 1}: parse failure ({exc})')
                if print_step_trace:
                    print(f'[Episode Improvement] pass {pass_idx + 1}: parse failure -> {exc}')
                continue

            candidate_note = f'pass={pass_idx + 1}: candidate_makespan={candidate_makespan:.1f}, baseline={best_makespan:.1f}'
            if candidate_feasible and candidate_makespan < best_makespan:
                best_makespan = float(candidate_makespan)
                best_traces = candidate_traces
                best_step_records = candidate_records
                improvement_notes.append(candidate_note + ' [accepted]')
                if print_step_trace:
                    print(f'[Episode Improvement] pass {pass_idx + 1}: accepted {candidate_makespan:.1f} < {best_makespan:.1f}')
            else:
                improvement_notes.append(candidate_note + ' [rejected]')
                if print_step_trace:
                    print(f'[Episode Improvement] pass {pass_idx + 1}: rejected')

    if print_step_trace:
        _print_step_trace(best_step_records)
        for note in improvement_notes:
            print('  improvement_note:', note)

    return float(best_makespan), True, best_traces

def force_safe_train_attention_backend(model):
    changed = []
    seen = set()
    for module in model.modules():
        cfg = getattr(module, 'config', None)
        if cfg is None:
            continue
        cfg_id = id(cfg)
        if cfg_id in seen:
            continue
        seen.add(cfg_id)
        for attr in ('_attn_implementation', 'attn_implementation'):
            if hasattr(cfg, attr):
                try:
                    setattr(cfg, attr, 'eager')
                    changed.append(f'{type(module).__name__}.{attr}=eager')
                except Exception:
                    pass
    backend_state = {'changed_configs': len(changed)}
    if torch.cuda.is_available() and hasattr(torch.backends, 'cuda'):
        try:
            if hasattr(torch.backends.cuda, 'enable_mem_efficient_sdp'):
                torch.backends.cuda.enable_mem_efficient_sdp(False)
            if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
                torch.backends.cuda.enable_flash_sdp(False)
            if hasattr(torch.backends.cuda, 'enable_math_sdp'):
                torch.backends.cuda.enable_math_sdp(True)
            backend_state.update({
                'flash_sdp': bool(torch.backends.cuda.flash_sdp_enabled()) if hasattr(torch.backends.cuda, 'flash_sdp_enabled') else None,
                'mem_efficient_sdp': bool(torch.backends.cuda.mem_efficient_sdp_enabled()) if hasattr(torch.backends.cuda, 'mem_efficient_sdp_enabled') else None,
                'math_sdp': bool(torch.backends.cuda.math_sdp_enabled()) if hasattr(torch.backends.cuda, 'math_sdp_enabled') else None,
            })
        except Exception as exc:
            backend_state['torch_sdp_note'] = str(exc)
    try:
        import unsloth.models.llama as unsloth_llama
        import unsloth.utils.attention_dispatch as unsloth_attention_dispatch
        def _force_sdpa_backend(*args, **kwargs):
            return unsloth_attention_dispatch.SDPA
        if hasattr(unsloth_attention_dispatch, 'HAS_XFORMERS'):
            unsloth_attention_dispatch.HAS_XFORMERS = False
            backend_state['unsloth_has_xformers'] = False
        if hasattr(unsloth_attention_dispatch, 'xformers_attention'):
            unsloth_attention_dispatch.xformers_attention = None
            backend_state['unsloth_xformers_attention'] = 'disabled'
        if hasattr(unsloth_attention_dispatch, 'select_attention_backend'):
            unsloth_attention_dispatch.select_attention_backend = _force_sdpa_backend
            backend_state['unsloth_backend_selector'] = 'sdpa'
        if hasattr(unsloth_llama, 'select_attention_backend'):
            unsloth_llama.select_attention_backend = _force_sdpa_backend
            backend_state['unsloth_llama_selector'] = 'sdpa'
        if hasattr(unsloth_llama, 'HAS_XFORMERS'):
            unsloth_llama.HAS_XFORMERS = False
        if hasattr(unsloth_llama, 'xformers_attention'):
            unsloth_llama.xformers_attention = None
    except Exception as exc:
        backend_state['unsloth_patch_note'] = str(exc)
    return backend_state

def compute_log_prob_mean(
    model,
    sequence_ids: torch.Tensor,
    prompt_len: int,
    action_token_id: int,
    device: torch.device,
    require_grad: bool,
) -> torch.Tensor:
    """
    Compute log-probability of the first generated action token only.
    """
    seq = sequence_ids[:prompt_len].unsqueeze(0).to(device)
    action_token_id = int(action_token_id)
    forward_kwargs = {
        'input_ids': seq,
        'use_cache': False,
        'num_logits_to_keep': 1,
    }
    if hasattr(torch.nn, 'attention') and hasattr(torch.nn.attention, 'sdpa_kernel') and hasattr(torch.nn.attention, 'SDPBackend'):
        sdp_ctx = torch.nn.attention.sdpa_kernel([torch.nn.attention.SDPBackend.MATH])
    else:
        sdp_ctx = (
            torch.backends.cuda.sdp_kernel(enable_flash=False, enable_mem_efficient=False, enable_math=True)
            if torch.cuda.is_available() and hasattr(torch.backends, 'cuda') and hasattr(torch.backends.cuda, 'sdp_kernel')
            else contextlib.nullcontext()
        )

    if require_grad:
        with sdp_ctx:
            outputs = model(**forward_kwargs)
    else:
        with torch.no_grad():
            with sdp_ctx:
                outputs = model(**forward_kwargs)

    logits = outputs.logits[:, -1, :]
    log_probs = F.log_softmax(logits, dim=-1)
    return log_probs[0, action_token_id]

def reinforce_step(
    batch: EpisodeBatch,
    optimizer: AdamW,
) -> float:
    """Perform a single REINFORCE update over collected samples."""

    advantages = batch.advantages
    if advantages.numel() > 1:
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    advantages = advantages.detach()
    loss = -(batch.log_probs * advantages).sum()

    optimizer.zero_grad()
    loss.backward()
    params = []
    for group in optimizer.param_groups:
        params.extend(group["params"])
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    optimizer.step()

    return loss.item()

def grpo_step(
    samples: List[TrajectorySample],
    model,
    optimizer: AdamW,
    device: torch.device,
    clip_epsilon: float = 0.2,
    grpo_epochs: int = 1,
    kl_coef: float = 0.0,
    update_batch_size: int = 4,
) -> Tuple[float, float]:
    """
    PPO-style clipped policy update with group-normalized advantages.
    """
    if not samples:
        return 0.0, 0.0

    old_log_probs = torch.stack([s.old_log_prob for s in samples]).to(device).detach()
    advantages = torch.tensor([s.advantage for s in samples], dtype=torch.float32, device=device)
    total_samples = int(len(samples))
    micro_batch_size = max(1, int(update_batch_size))

    total_loss = 0.0
    total_kl = 0.0
    for _ in range(grpo_epochs):
        optimizer.zero_grad()
        epoch_loss_value = 0.0
        epoch_kl_value = 0.0
        for start in range(0, total_samples, micro_batch_size):
            end = min(start + micro_batch_size, total_samples)
            chunk = samples[start:end]
            chunk_old = old_log_probs[start:end]
            chunk_adv = advantages[start:end]
            current_log_probs = []
            for s in chunk:
                log_prob = compute_log_prob_mean(
                    model=model,
                    sequence_ids=s.sequence_ids,
                    prompt_len=s.prompt_len,
                    action_token_id=s.action_token_id,
                    device=device,
                    require_grad=True,
                )
                current_log_probs.append(log_prob)
            current_log_probs = torch.stack(current_log_probs)

            log_ratio = current_log_probs - chunk_old
            ratio = torch.exp(torch.clamp(log_ratio, -20, 20))
            unclipped_obj = ratio * chunk_adv
            clipped_obj = torch.clamp(ratio, 1.0 - clip_epsilon, 1.0 + clip_epsilon) * chunk_adv
            policy_loss = -torch.mean(torch.min(unclipped_obj, clipped_obj))

            approx_kl = torch.mean((ratio - 1.0) - torch.log(ratio + 1e-8))
            loss = policy_loss + kl_coef * approx_kl
            scaled_loss = loss * (float(end - start) / float(total_samples))
            scaled_loss.backward()
            weight = float(end - start) / float(total_samples)
            epoch_loss_value += float(loss.item()) * weight
            epoch_kl_value += float(approx_kl.item()) * weight
            del current_log_probs, log_ratio, ratio, unclipped_obj, clipped_obj, policy_loss, approx_kl, loss, scaled_loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        params = []
        for group in optimizer.param_groups:
            params.extend(group["params"])
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optimizer.step()

        total_loss += float(epoch_loss_value)
        total_kl += float(epoch_kl_value)

    return total_loss / grpo_epochs, total_kl / grpo_epochs

def build_bopo_step_pairs(
    group_rollouts: List[Dict],
    rng: np.random.Generator,
    min_relative_gap: float = 0.0,
    max_pairs_per_group: int = 256,
    max_step_pairs_per_pair: int = 32,
    pair_mode: str = "shared_prefix",
) -> List[BOPOStepPair]:
    """
    Build BOPO preference pairs from a rollout group.

    Strategy:
      1. Keep feasible rollouts only.
      2. Sort by makespan (smaller is better), anchor best rollout.
      3. Pair best vs each loser if relative gap >= threshold.
      4. For each episode pair, either:
         - sample aligned step indices (`aligned`)
         - or keep only the first divergence step after shared prefix (`shared_prefix`)
    """
    feasible_rollouts = [
        r
        for r in group_rollouts
        if bool(r.get("feasible"))
        and math.isfinite(float(r.get("makespan", float("inf"))))
        and len(r.get("traces", [])) > 0
    ]
    if len(feasible_rollouts) < 2:
        return []

    feasible_rollouts.sort(key=lambda x: float(x["makespan"]))
    winner = feasible_rollouts[0]
    winner_ms = float(winner["makespan"])
    winner_traces = winner["traces"]

    resolved_pair_mode = str(pair_mode).strip().lower()
    if resolved_pair_mode not in {"aligned", "shared_prefix"}:
        raise ValueError(f"Unsupported pair_mode={pair_mode}")

    def _first_divergence_index(w_traces, l_traces):
        n_steps_local = min(len(w_traces), len(l_traces))
        for idx in range(n_steps_local):
            if int(w_traces[idx].chosen_job) != int(l_traces[idx].chosen_job):
                return idx
        return None

    pairs: List[BOPOStepPair] = []
    for loser in feasible_rollouts[1:]:
        loser_ms = float(loser["makespan"])
        if loser_ms <= winner_ms:
            continue
        rel_gap = (loser_ms - winner_ms) / max(loser_ms, 1.0)
        if rel_gap < float(min_relative_gap):
            continue

        loser_traces = loser["traces"]
        n_steps = min(len(winner_traces), len(loser_traces))
        if n_steps <= 0:
            continue

        if resolved_pair_mode == "shared_prefix":
            div_idx = _first_divergence_index(winner_traces, loser_traces)
            if div_idx is None:
                continue
            indices = [int(div_idx)]
        else:
            indices = list(range(n_steps))
            max_steps = int(max_step_pairs_per_pair)
            if max_steps > 0 and n_steps > max_steps:
                picked = rng.choice(n_steps, size=max_steps, replace=False)
                indices = [int(x) for x in picked.tolist()]

        for step_idx in indices:
            w_trace = winner_traces[step_idx]
            l_trace = loser_traces[step_idx]
            pairs.append(
                BOPOStepPair(
                    winner_sequence_ids=w_trace.sequence_ids,
                    winner_prompt_len=int(w_trace.prompt_len),
                    winner_action_token_id=int(w_trace.action_token_id),
                    loser_sequence_ids=l_trace.sequence_ids,
                    loser_prompt_len=int(l_trace.prompt_len),
                    loser_action_token_id=int(l_trace.action_token_id),
                    relative_gap=float(rel_gap),
                    winner_makespan=float(winner_ms),
                    loser_makespan=float(loser_ms),
                )
            )

    max_pairs = int(max_pairs_per_group)
    if max_pairs > 0 and len(pairs) > max_pairs:
        picked = rng.choice(len(pairs), size=max_pairs, replace=False)
        pairs = [pairs[int(i)] for i in picked.tolist()]

    return pairs

def bopo_step(
    pairs: List[BOPOStepPair],
    model,
    optimizer: AdamW,
    device: torch.device,
    beta: float = 2.0,
    gap_scale: float = 3.0,
    margin: float = 0.0,
) -> Tuple[float, float, int]:
    """
    BOPO-style pairwise preference optimization.

    loss = -log sigma( beta * (1 + gap_scale * rel_gap) * (logp_win - logp_lose - margin) )
    """
    if not pairs:
        return 0.0, 0.0, 0

    total_loss = 0.0
    total_gap = 0.0
    updates = 0

    for p in pairs:
        try:
            lp_w = compute_log_prob_mean(
                model=model,
                sequence_ids=p.winner_sequence_ids,
                prompt_len=p.winner_prompt_len,
                action_token_id=p.winner_action_token_id,
                device=device,
                require_grad=True,
            )
            lp_l = compute_log_prob_mean(
                model=model,
                sequence_ids=p.loser_sequence_ids,
                prompt_len=p.loser_prompt_len,
                action_token_id=p.loser_action_token_id,
                device=device,
                require_grad=True,
            )

            rel_gap = max(0.0, float(p.relative_gap))
            scaled_beta = float(beta) * (1.0 + float(gap_scale) * rel_gap)
            pref_logit = scaled_beta * (lp_w - lp_l - float(margin))
            loss = -F.logsigmoid(pref_logit)

            optimizer.zero_grad()
            loss.backward()
            params = []
            for group in optimizer.param_groups:
                params.extend(group["params"])
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()

            total_loss += float(loss.item())
            total_gap += rel_gap
            updates += 1

            del lp_w, lp_l, loss, pref_logit
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower():
                optimizer.zero_grad(set_to_none=True)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise

    return total_loss / max(updates, 1), total_gap / max(updates, 1), updates

def resolve_upload_dir(output_dir, upload_source='latest_checkpoint', checkpoint_tag=None):
    output_dir = Path(output_dir)
    source = str(upload_source or 'latest_checkpoint')
    if source == 'latest_checkpoint':
        checkpoints = sorted(
            output_dir.glob('checkpoint-*'),
            key=lambda p: p.name,
        )
        if not checkpoints:
            raise FileNotFoundError(f'no checkpoint-* directories found under {output_dir}')
        return checkpoints[-1]
    if source == 'final':
        p = output_dir / 'final'
        if not p.exists():
            raise FileNotFoundError(f'final folder not found: {p}')
        return p
    if checkpoint_tag:
        p = output_dir / str(checkpoint_tag)
        if not p.exists():
            raise FileNotFoundError(f'checkpoint folder not found: {p}')
        return p
    raise ValueError(f'Unsupported upload_source={upload_source!r}')


# ---------------------------------------------------------------------------
# Official Unsloth GRPO helpers
# ---------------------------------------------------------------------------
GRPO_ACTION_LINE_PATTERN = re.compile(r'^(<\s*[aAsS]\s*\d+\s*>)\s*\|')
GRPO_CMAX_PATTERN = re.compile(r'cmax\s*:\s*(-?\d+)\s*->\s*(-?\d+)', re.IGNORECASE)
GRPO_DELTA_CMAX_PATTERN = re.compile(r'delta_cmax\s*=\s*(-?\d+)', re.IGNORECASE)
GRPO_EST_END_PATTERN = re.compile(r'est_end\s*=\s*(-?\d+)', re.IGNORECASE)
GRPO_PROC_TIME_PATTERN = re.compile(r'processing\s+time\s*=\s*(-?\d+)', re.IGNORECASE)
GRPO_REM_RATIO_PATTERN = re.compile(r'rem_work_after_ratio\s*=\s*([0-9]*\.?[0-9]+)', re.IGNORECASE)


def _completion_to_text(completion) -> str:
    if completion is None:
        return ''
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        return str(completion.get('content', ''))
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, dict):
                parts.append(str(item.get('content', '')))
            else:
                parts.append(str(item))
        return '\n'.join(parts)
    return str(completion)


def _normalize_action_code_list(action_codes) -> List[str]:
    out = []
    for code in list(action_codes or []):
        text = str(code).strip()
        parsed = parse_action_code(text, code_width=int(CFG.get('action_code_width', 4)))
        out.append(str(parsed or text))
    return out


def _extract_target_action_code(target_text: str) -> Optional[str]:
    parsed = parse_action_code(str(target_text or ''), code_width=int(CFG.get('action_code_width', 4)))
    if parsed is not None:
        return str(parsed)
    stripped = str(target_text or '').strip()
    return stripped if stripped else None


def extract_proxy_action_metrics_from_state_text(state_text: str) -> Dict[str, Dict[str, float]]:
    metrics: Dict[str, Dict[str, float]] = {}
    for raw_line in str(state_text or '').splitlines():
        line = raw_line.strip()
        if not line.startswith('<'):
            continue
        code = parse_action_code(line, code_width=int(CFG.get('action_code_width', 4)))
        if code is None:
            continue
        cmax_match = GRPO_CMAX_PATTERN.search(line)
        delta_match = GRPO_DELTA_CMAX_PATTERN.search(line)
        est_end_match = GRPO_EST_END_PATTERN.search(line)
        proc_match = GRPO_PROC_TIME_PATTERN.search(line)
        rem_ratio_match = GRPO_REM_RATIO_PATTERN.search(line)
        metrics[str(code)] = {
            'action_code': str(code),
            'cmax_after': float(int(cmax_match.group(2))) if cmax_match else float(10**12),
            'delta_cmax': float(int(delta_match.group(1))) if delta_match else float(10**12),
            'est_end': float(int(est_end_match.group(1))) if est_end_match else float(10**12),
            'proc_time': float(int(proc_match.group(1))) if proc_match else float(10**12),
            'rem_work_after_ratio': float(rem_ratio_match.group(1)) if rem_ratio_match else 1e9,
        }
    ordered = sorted(
        metrics.values(),
        key=lambda x: (
            float(x['cmax_after']),
            float(x['delta_cmax']),
            float(x['est_end']),
            float(x['proc_time']),
            str(x['action_code']),
        ),
    )
    if not ordered:
        return metrics
    best_after = float(ordered[0]['cmax_after'])
    best_delta = float(ordered[0]['delta_cmax'])
    max_after_gap = max(float(item['cmax_after']) - best_after for item in ordered)
    max_delta_gap = max(float(item['delta_cmax']) - best_delta for item in ordered)
    denom_rank = max(len(ordered) - 1, 1)
    for rank, item in enumerate(ordered):
        after_gap = max(0.0, float(item['cmax_after']) - best_after)
        delta_gap = max(0.0, float(item['delta_cmax']) - best_delta)
        item['rank'] = float(rank)
        item['rank_score'] = 1.0 if len(ordered) == 1 else 1.0 - float(rank) / float(denom_rank)
        item['cmax_gap_score'] = 1.0 - (after_gap / max(max_after_gap, 1.0))
        item['delta_gap_score'] = 1.0 - (delta_gap / max(max_delta_gap, 1.0))
    return {str(item['action_code']): dict(item) for item in ordered}


def build_unsloth_grpo_prompt(state_text: str):
    return [
        {
            'role': 'system',
            'content': 'You are solving JSSP step-by-step. Output exactly one feasible action code like <A1234>. Do not explain your answer.',
        },
        {
            'role': 'user',
            'content': str(state_text),
        },
    ]


def resolve_grpo_step_dataset_path(cfg, hf_token: str = '') -> str:
    if cfg.get('grpo_dataset_path'):
        return os.path.expanduser(str(cfg['grpo_dataset_path']))
    source = str(cfg.get('grpo_dataset_source', 'hf')).strip().lower()
    if source == 'local':
        if str(cfg.get('env_mode', 'serial')).strip().lower() == 'dispatch':
            return os.path.expanduser(str(cfg['grpo_step_dataset_local_path_dispatch']))
        return os.path.expanduser(str(cfg['grpo_step_dataset_local_path']))
    if source != 'hf':
        raise ValueError("grpo_dataset_source must be 'hf' or 'local'.")
    if str(cfg.get('env_mode', 'serial')).strip().lower() == 'dispatch':
        repo_id = str(cfg['grpo_step_dataset_hf_repo_dispatch'])
        filename = str(cfg['grpo_step_dataset_hf_file_dispatch'])
    else:
        repo_id = str(cfg['grpo_step_dataset_hf_repo'])
        filename = str(cfg['grpo_step_dataset_hf_file'])
    return hf_hub_download(repo_id=repo_id, repo_type='dataset', filename=filename, token=hf_token or None)


def build_unsloth_grpo_step_dataset(cfg, hf_token: str = ''):
    dataset_path = resolve_grpo_step_dataset_path(cfg, hf_token=hf_token)
    ds = load_dataset('json', data_files=dataset_path)['train']
    raw_rows = len(ds)
    exact_jobs = cfg.get('rl_num_jobs')
    exact_machines = cfg.get('rl_num_machines')
    min_jobs = cfg.get('min_rl_num_jobs')
    min_machines = cfg.get('min_rl_num_machines')
    min_feasible = int(cfg.get('grpo_min_feasible_actions', 2) or 2)
    raw_size_counter = Counter()
    filtered_size_counter = Counter()
    records = []
    for row in ds:
        n_jobs = int(row.get('num_jobs', 0) or 0)
        n_machines = int(row.get('num_machines', 0) or 0)
        num_feasible = int(row.get('num_feasible_actions', len(row.get('feasible_action_codes', []) or [])) or 0)
        raw_size_counter[(n_jobs, n_machines)] += 1
        keep = True
        if exact_jobs is not None and n_jobs != int(exact_jobs):
            keep = False
        if exact_machines is not None and n_machines != int(exact_machines):
            keep = False
        if min_jobs is not None and n_jobs < int(min_jobs):
            keep = False
        if min_machines is not None and n_machines < int(min_machines):
            keep = False
        if num_feasible < min_feasible:
            keep = False
        if not keep:
            continue
        state_text = str(row.get('state_text', '')).strip()
        feasible_action_codes = _normalize_action_code_list(row.get('feasible_action_codes', []) or [])
        if not state_text or not feasible_action_codes:
            continue
        proxy_metrics = extract_proxy_action_metrics_from_state_text(state_text)
        target_action_code = _extract_target_action_code(row.get('target_text', ''))
        records.append({
            'prompt': build_unsloth_grpo_prompt(state_text),
            'state_text': state_text,
            'feasible_action_codes': list(feasible_action_codes),
            'proxy_metrics_json': json.dumps(proxy_metrics, ensure_ascii=False),
            'target_action_code': target_action_code,
            'instance_id': str(row.get('instance_id', '')),
            'step_idx': int(row.get('step_idx', 0) or 0),
            'num_jobs': n_jobs,
            'num_machines': n_machines,
            'num_feasible_actions': num_feasible,
        })
        filtered_size_counter[(n_jobs, n_machines)] += 1
    if cfg.get('grpo_shuffle_seed') is not None:
        rng_local = random.Random(int(cfg['grpo_shuffle_seed']))
        rng_local.shuffle(records)
    max_rows = cfg.get('grpo_max_dataset_rows')
    if max_rows is not None:
        max_rows = int(max_rows)
        if max_rows > 0:
            records = records[:max_rows]
    print('official grpo dataset path:', dataset_path)
    print('official grpo raw rows:', raw_rows)
    print('official grpo filtered rows:', len(records))
    print('official grpo size_dist_top_raw:', raw_size_counter.most_common(10))
    print('official grpo size_dist_top_filtered:', filtered_size_counter.most_common(10))
    print('official grpo filter config:', {
        'rl_num_jobs': exact_jobs,
        'rl_num_machines': exact_machines,
        'min_rl_num_jobs': min_jobs,
        'min_rl_num_machines': min_machines,
        'grpo_min_feasible_actions': min_feasible,
        'grpo_max_dataset_rows': max_rows,
    })
    if not records:
        raise ValueError('No GRPO step rows remain after filtering.')
    return Dataset.from_list(records), dataset_path


def build_unsloth_grpo_reward_functions(cfg):
    valid_weight = float(cfg.get('grpo_reward_valid_weight', 1.0))
    proxy_weight = float(cfg.get('grpo_reward_proxy_weight', 1.0))
    teacher_weight = float(cfg.get('grpo_reward_teacher_weight', 0.25))

    def reward_valid_action(prompts=None, completions=None, feasible_action_codes=None, **kwargs):
        feasible_lists = feasible_action_codes or kwargs.get('feasible_action_codes') or []
        rewards = []
        for completion, feasible in zip(completions or [], feasible_lists):
            code = parse_action_code(_completion_to_text(completion), code_width=int(cfg.get('action_code_width', 4)))
            feasible_set = set(_normalize_action_code_list(feasible))
            if code is None:
                rewards.append(-1.0 * valid_weight)
            elif str(code) in feasible_set:
                rewards.append(1.0 * valid_weight)
            else:
                rewards.append(-0.5 * valid_weight)
        return rewards

    def reward_proxy_quality(prompts=None, completions=None, proxy_metrics_json=None, feasible_action_codes=None, **kwargs):
        metric_rows = proxy_metrics_json or kwargs.get('proxy_metrics_json') or []
        rewards = []
        for completion, metrics_blob in zip(completions or [], metric_rows):
            code = parse_action_code(_completion_to_text(completion), code_width=int(cfg.get('action_code_width', 4)))
            try:
                metrics = json.loads(metrics_blob or '{}')
            except Exception:
                metrics = {}
            if code is None or str(code) not in metrics:
                rewards.append(-1.0 * proxy_weight)
                continue
            chosen = metrics[str(code)]
            score = (
                0.60 * float(chosen.get('cmax_gap_score', 0.0)) +
                0.25 * float(chosen.get('delta_gap_score', 0.0)) +
                0.15 * float(chosen.get('rank_score', 0.0))
            )
            rewards.append(float(score) * proxy_weight)
        return rewards

    def reward_teacher_action(prompts=None, completions=None, target_action_code=None, **kwargs):
        targets = target_action_code or kwargs.get('target_action_code') or []
        rewards = []
        for completion, target in zip(completions or [], targets):
            code = parse_action_code(_completion_to_text(completion), code_width=int(cfg.get('action_code_width', 4)))
            rewards.append(float(teacher_weight) if code is not None and target is not None and str(code) == str(target) else 0.0)
        return rewards

    reward_valid_action.__name__ = 'reward_valid_action'
    reward_proxy_quality.__name__ = 'reward_proxy_quality'
    reward_teacher_action.__name__ = 'reward_teacher_action'
    return [reward_valid_action, reward_proxy_quality, reward_teacher_action]


## 3) RL 학습 실행


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from collections import Counter
import csv
import re

def build_action_special_tokens(code_width: int = 4, code_start: int = 1, code_cap: int = 9999, prefix: str = ACTION_TOKEN_PREFIX):
    return [
        format_action_code(code_idx, code_width=code_width, prefix=prefix)
        for code_idx in range(int(code_start), int(code_cap) + 1)
    ]

def ensure_action_special_tokens(tokenizer, model=None, code_width: int = 4, code_start: int = 1, code_cap: int = 9999, prefix: str = ACTION_TOKEN_PREFIX):
    action_tokens = build_action_special_tokens(
        code_width=code_width,
        code_start=code_start,
        code_cap=code_cap,
        prefix=prefix,
    )
    existing = set(getattr(tokenizer, 'additional_special_tokens', []) or [])
    missing = [tok for tok in action_tokens if tok not in existing]
    added = 0
    if missing:
        added = int(tokenizer.add_special_tokens({'additional_special_tokens': missing}))

    if model is not None:
        target_size = len(tokenizer)
        input_size = int(model.get_input_embeddings().weight.shape[0])
        if input_size != target_size:
            model.resize_token_embeddings(target_size)

    return {
        'num_added_tokens': int(added),
        'vocab_size': int(len(tokenizer)),
        'action_token_count': int(len(action_tokens)),
    }

def _resolve_checkpoint_tag(repo_id, checkpoint_tag=None, token=None):
    if not checkpoint_tag:
        return None
    expanded = os.path.expanduser(str(repo_id))
    if os.path.exists(expanded):
        return None
    api = HfApi(token=token)
    files = set(api.list_repo_files(repo_id=repo_id, repo_type='model'))
    tag = str(checkpoint_tag)
    if any(path.startswith(f'{tag}/') for path in files):
        return tag
    raise FileNotFoundError(f'checkpoint tag not found in repo {repo_id!r}: {tag!r}')


def _resolve_model_source(model_path, checkpoint_tag=None, token=None):
    expanded = os.path.expanduser(str(model_path))
    if os.path.exists(expanded):
        return expanded
    resolved_checkpoint = _resolve_checkpoint_tag(expanded, checkpoint_tag, token=token)
    if resolved_checkpoint is None:
        return expanded
    snapshot_dir = snapshot_download(
        repo_id=expanded,
        repo_type='model',
        allow_patterns=[f'{resolved_checkpoint}/*'],
        token=token,
    )
    local_checkpoint_dir = os.path.join(snapshot_dir, resolved_checkpoint)
    if os.path.exists(local_checkpoint_dir):
        return local_checkpoint_dir
    for probe_file in ('adapter_config.json', 'adapter_model.safetensors'):
        try:
            local_file = hf_hub_download(
                repo_id=expanded,
                repo_type='model',
                filename=f'{resolved_checkpoint}/{probe_file}',
                token=token,
            )
            return os.path.dirname(local_file)
        except Exception:
            pass
    raise FileNotFoundError(f'checkpoint folder not found after download: {local_checkpoint_dir}')


def _is_adapter_source(path_or_repo, token=None):
    if not path_or_repo:
        return False
    if os.path.exists(path_or_repo):
        return os.path.exists(os.path.join(path_or_repo, 'adapter_config.json'))
    try:
        api = HfApi(token=token)
        files = set(api.list_repo_files(repo_id=path_or_repo, repo_type='model'))
    except Exception:
        return False
    return 'adapter_config.json' in files


def _maybe_load_policy_adapter(model, policy_model_path, token=None):
    if not policy_model_path:
        return False
    if not hasattr(model, 'load_adapter'):
        raise RuntimeError('Policy adapter path was provided, but model.load_adapter is unavailable.')
    if os.path.exists(policy_model_path):
        model.load_adapter(policy_model_path, adapter_name='default', adapter_kwargs={'local_files_only': True})
    else:
        model.load_adapter(policy_model_path, adapter_name='default', token=token)
    if hasattr(model, 'set_adapter'):
        model.set_adapter('default')
    return True

def _summarize_trainable_parameters(model):
    trainable = 0
    total = 0
    for param in model.parameters():
        n = int(param.numel())
        total += n
        if param.requires_grad:
            trainable += n
    ratio = (float(trainable) / float(total)) if total > 0 else 0.0
    return trainable, total, ratio


def _validate_rl_update_mode(model, update_mode: str, model_path: str):
    trainable, total, ratio = _summarize_trainable_parameters(model)
    print(f'[Info] RL trainable params: {trainable:,} / {total:,} ({ratio * 100:.2f}%)')
    if update_mode == 'adapter_only':
        if trainable <= 0:
            raise ValueError(f'RL update mode is adapter_only, but no trainable parameters were found. model_path={model_path!r}')
    elif update_mode == 'full':
        if trainable <= 0:
            raise ValueError(f'RL update mode is full, but no trainable parameters were found. model_path={model_path!r}')
    else:
        raise ValueError(f'Unsupported rl_update_mode={update_mode}')
    return trainable, total, ratio


if CFG.get('hf_token'):
    login(token=CFG['hf_token'], add_to_git_credential=False)

model_path = CFG['model_path'] if CFG['model_path'] else MODEL_MAP[CFG['model_type']]
if CFG['rl_algo'] == 'unsloth_grpo':
    PatchFastRL('GRPO', FastLanguageModel)
    print('official RL backend: unsloth GRPO')

resolved_model_path = _resolve_model_source(
    model_path,
    checkpoint_tag=CFG.get('model_checkpoint_tag'),
    token=CFG.get('hf_token'),
)
print('loading model:', resolved_model_path)
is_adapter = _is_adapter_source(resolved_model_path, token=CFG.get('hf_token'))
base_load_path = MODEL_MAP[CFG['model_type']] if is_adapter else resolved_model_path
print('base load path:', base_load_path)

load_kwargs = {
    'model_name': base_load_path,
    'max_seq_length': CFG['max_seq_length'],
    'load_in_4bit': CFG['load_in_4bit'],
    'dtype': torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16,
    'local_files_only': False,
}
if CFG['rl_algo'] == 'unsloth_grpo':
    load_kwargs['fast_inference'] = bool(CFG.get('grpo_use_fast_inference', False))
    load_kwargs['gpu_memory_utilization'] = float(CFG.get('grpo_gpu_memory_utilization', 0.6))
model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
token_install = ensure_action_special_tokens(
    tokenizer=tokenizer,
    model=model,
    code_width=int(CFG.get('action_code_width', 4)),
    code_cap=int(CFG.get('action_code_cap', 9999)),
)
print('action token install:', token_install)
if is_adapter:
    _maybe_load_policy_adapter(model, resolved_model_path, token=CFG.get('hf_token'))
    print('policy adapter loaded')
if hasattr(model, 'for_training'):
    model.for_training()
if CFG['rl_algo'] == 'unsloth_grpo':
    print('rl attention backend: official unsloth grpo trainer path')
else:
    attn_backend_info = force_safe_train_attention_backend(model)
    print('rl attention backend:', attn_backend_info)
model.train()
_validate_rl_update_mode(model, CFG.get('rl_update_mode', 'adapter_only'), resolved_model_path)

output_dir = Path(CFG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

if CFG['rl_algo'] == 'unsloth_grpo':
    try:
        from unsloth import UnslothGRPOConfig as GRPOConfig
    except Exception:
        from trl import GRPOConfig
    from trl import GRPOTrainer

    grpo_dataset, grpo_dataset_path = build_unsloth_grpo_step_dataset(CFG, hf_token=HF_TOKEN)
    reward_funcs = build_unsloth_grpo_reward_functions(CFG)
    max_completion_length = int(CFG.get('grpo_max_completion_length', CFG.get('step_action_max_new_tokens', 8)) or 8)
    max_prompt_length = int(CFG.get('grpo_max_prompt_length', max(256, CFG['max_seq_length'] - max_completion_length - 32)) or 4096)
    max_prompt_length = min(max_prompt_length, int(CFG['max_seq_length']) - max_completion_length - 8)
    if max_prompt_length <= 0:
        raise ValueError('grpo_max_prompt_length must be smaller than max_seq_length - max_completion_length')

    grpo_num_train_epochs = CFG.get('grpo_num_train_epochs')
    grpo_args = GRPOConfig(
        output_dir=str(output_dir),
        learning_rate=float(CFG['learning_rate']),
        per_device_train_batch_size=int(CFG.get('grpo_per_device_train_batch_size', 1)),
        gradient_accumulation_steps=int(CFG.get('grpo_gradient_accumulation_steps', 1)),
        num_generations=int(CFG.get('grpo_num_generations', CFG.get('group_size', 4))),
        max_prompt_length=int(max_prompt_length),
        max_completion_length=int(max_completion_length),
        num_train_epochs=float(CFG.get('epochs', 1) if grpo_num_train_epochs in (None, '') else grpo_num_train_epochs),
        max_steps=int(CFG.get('grpo_max_steps', -1)),
        logging_steps=int(CFG.get('grpo_logging_steps', 5)),
        save_steps=int(CFG.get('grpo_save_steps', 100)),
        bf16=bool(CFG['dtype'] == 'bfloat16' and torch.cuda.is_available() and is_bfloat16_supported()),
        fp16=bool(CFG['dtype'] != 'bfloat16'),
        remove_unused_columns=False,
        report_to=CFG.get('grpo_report_to', 'none'),
        optim=str(CFG.get('grpo_optim', 'adamw_8bit')),
        weight_decay=float(CFG.get('grpo_weight_decay', 0.01)),
        warmup_ratio=float(CFG.get('grpo_warmup_ratio', 0.05)),
        lr_scheduler_type=str(CFG.get('grpo_lr_scheduler_type', 'linear')),
        max_grad_norm=float(CFG.get('grpo_max_grad_norm', 1.0)),
        seed=int(CFG.get('seed', 42)),
    )

    print('official grpo config:', {
        'train_rows': len(grpo_dataset),
        'num_generations': int(CFG.get('grpo_num_generations', CFG.get('group_size', 4))),
        'num_train_epochs': float(CFG.get('epochs', 1) if grpo_num_train_epochs in (None, '') else grpo_num_train_epochs),
        'max_steps': int(CFG.get('grpo_max_steps', -1)),
        'max_prompt_length': int(max_prompt_length),
        'max_completion_length': int(max_completion_length),
    })

    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=reward_funcs,
        args=grpo_args,
        train_dataset=grpo_dataset,
    )
    resume_ckpt = CFG.get('resume_from_checkpoint') or None
    train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
    trainer.save_model(str(output_dir))
    tokenizer.save_pretrained(str(output_dir))

    grpo_log_history = list(getattr(trainer.state, 'log_history', []) or [])
    grpo_log_path = output_dir / 'grpo_log_history.csv'
    if grpo_log_history:
        _write_history_csv(grpo_log_path, grpo_log_history)
        print('grpo_log_history_path:', grpo_log_path)
    else:
        print('No GRPO log history found.')
    print('official grpo dataset path:', grpo_dataset_path)
    print('official grpo train rows:', len(grpo_dataset))
    print('training done')
    print('output_dir:', output_dir)
else:
    optimizer = AdamW(model.parameters(), lr=CFG['learning_rate'])

    rng = np.random.default_rng(CFG['seed'])
    if CFG['use_random_problems']:
        dataset_split = None
    else:
        if CFG.get('dataset_path'):
            dataset_path = os.path.expanduser(CFG['dataset_path'])
        elif CFG['dataset_source'] == 'hf':
            dataset_path = hf_hub_download(
                repo_id=CFG['dataset_hf_repo'],
                repo_type='dataset',
                filename=CFG['dataset_hf_file'],
                token=HF_TOKEN,
            )
        elif CFG['dataset_source'] == 'local':
            dataset_path = os.path.expanduser(CFG['dataset_local_path'])
        else:
            raise ValueError("CFG['dataset_source'] must be 'hf' or 'local'")
    
        print('dataset path:', dataset_path)
        ds = load_dataset('json', data_files=dataset_path)
        dataset_split = ds['train']
    
        def _extract_problem_size(example):
            n_jobs = example.get('num_jobs')
            n_machines = example.get('num_machines')
            if n_jobs is not None and n_machines is not None:
                return int(n_jobs), int(n_machines)
            matrix = example.get('matrix')
            if not isinstance(matrix, str) or not matrix.strip():
                raise ValueError('RL dataset example is missing num_jobs/num_machines and matrix header.')
            first_line = next((ln.strip() for ln in matrix.splitlines() if ln.strip()), None)
            if not first_line:
                raise ValueError('RL dataset matrix is empty.')
            parts = first_line.split()
            if len(parts) < 2:
                raise ValueError(f'Failed to parse problem size from matrix header: {first_line!r}')
            return int(parts[0]), int(parts[1])
    
        exact_jobs = CFG.get('rl_num_jobs')
        exact_machines = CFG.get('rl_num_machines')
        min_jobs = CFG.get('min_rl_num_jobs')
        min_machines = CFG.get('min_rl_num_machines')
        raw_rows = len(dataset_split)
        raw_size_counter = Counter()
        filtered_size_counter = Counter()
        keep_indices = []
    
        for idx, example in enumerate(dataset_split):
            n_jobs, n_machines = _extract_problem_size(example)
            size_key = (int(n_jobs), int(n_machines))
            raw_size_counter[size_key] += 1
    
            keep = True
            if exact_jobs is not None and int(n_jobs) != int(exact_jobs):
                keep = False
            if exact_machines is not None and int(n_machines) != int(exact_machines):
                keep = False
            if min_jobs is not None and int(n_jobs) < int(min_jobs):
                keep = False
            if min_machines is not None and int(n_machines) < int(min_machines):
                keep = False
    
            if keep:
                keep_indices.append(int(idx))
                filtered_size_counter[size_key] += 1
    
        if len(keep_indices) != raw_rows:
            dataset_split = dataset_split.select(keep_indices)
    
        print('rl dataset raw rows:', raw_rows)
        print('rl dataset filtered rows:', len(dataset_split))
        print('rl dataset kept ratio:', (len(dataset_split) / raw_rows) if raw_rows else 0.0)
        print('rl dataset size_dist_top_raw:', raw_size_counter.most_common(10))
        print('rl dataset size_dist_top_filtered:', filtered_size_counter.most_common(10))
        print('rl dataset filter config:', {
            'rl_num_jobs': exact_jobs,
            'rl_num_machines': exact_machines,
            'min_rl_num_jobs': min_jobs,
            'min_rl_num_machines': min_machines,
        })
    
        if len(dataset_split) == 0:
            raise ValueError('No RL training problems remain after applying size filters.')
    
    baseline_tracker = ExponentialBaseline(beta=CFG['baseline_beta'])
    output_dir = Path(CFG['output_dir'])
    output_dir.mkdir(parents=True, exist_ok=True)
    
    epoch_history = []
    episode_history = []
    epoch_metrics_path = output_dir / 'rl_epoch_metrics.csv'
    episode_metrics_path = output_dir / 'rl_episode_metrics.csv'
    REFERENCE_MAKESPAN_PATTERN = re.compile(r"Makespan:\s*(\d+(?:\.\d+)?)", re.IGNORECASE)
    
    def _extract_reference_makespan(example):
        if not isinstance(example, dict):
            return None
        for key in ('benchmark_makespan', 'optimal_makespan', 'optimum_makespan', 'real_makespan', 'makespan'):
            value = example.get(key)
            if value is None:
                continue
            try:
                return float(value)
            except Exception:
                pass
        output_text = example.get('output')
        if isinstance(output_text, str):
            match = REFERENCE_MAKESPAN_PATTERN.search(output_text)
            if match:
                try:
                    return float(match.group(1))
                except Exception:
                    return None
        return None
    
    def _safe_mean(values):
        vals = [float(v) for v in values if v is not None and math.isfinite(float(v))]
        if not vals:
            return None
        return float(np.mean(vals))
    
    def _safe_median(values):
        vals = [float(v) for v in values if v is not None and math.isfinite(float(v))]
        if not vals:
            return None
        return float(np.median(vals))
    
    def _write_history_csv(path_obj: Path, rows):
        if not rows:
            return
        fieldnames = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    fieldnames.append(key)
        with path_obj.open('w', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow(row)
    
    
    def _print_running_epoch_status(epoch_idx, episode_idx, cfg, *, best_makespans, gaps, pair_counts, feasible_counts, note=None):
        msg = {
            'epoch': int(epoch_idx + 1),
            'episode': int(episode_idx + 1),
            'avg_best_makespan': _safe_mean(best_makespans),
            'avg_gap_vs_reference': _safe_mean(gaps),
            'avg_pair_count': _safe_mean(pair_counts),
            'avg_feasible_rollouts': _safe_mean(feasible_counts),
            'note': note,
        }
        print('[Running]', msg)
    
    
    def _format_gap_pct(gap_value):
        if gap_value is None:
            return 'NA'
        try:
            return f"{float(gap_value) * 100.0:.2f}%"
        except Exception:
            return 'NA'
    
    def _print_episode_result(epoch_idx, episode_idx, total_episodes, *, algo, num_jobs, num_machines, reference_makespan, best_makespan, gap_vs_reference, pair_count=None, feasible_rollouts=None, note=None):
        print(
            f"[Episode {episode_idx + 1}/{int(total_episodes)}] epoch={epoch_idx + 1} algo={algo} size={int(num_jobs)}x{int(num_machines)} "
            f"opt_ms={reference_makespan if reference_makespan is not None else 'NA'} "
            f"best_ms={best_makespan if best_makespan is not None else 'NA'} "
            f"gap={_format_gap_pct(gap_vs_reference)} "
            f"pairs={pair_count if pair_count is not None else 'NA'} "
            f"feasible_rollouts={feasible_rollouts if feasible_rollouts is not None else 'NA'} "
            f"note={note if note is not None else '-'}"
        )
    
    for epoch in range(CFG['epochs']):
        grpo_samples = []
        bopo_pairs_epoch = []
        reinforce_updates = 0
        reinforce_loss_sum = 0.0
        bopo_updates = 0
        bopo_loss_total = 0.0
        bopo_gap_total = 0.0
    
        epoch_best_makespans = []
        epoch_reference_gaps = []
        epoch_pair_counts = []
        epoch_feasible_rollout_counts = []
        epoch_group_rewards = []
        epoch_advantages = []
        epoch_trace_losses = []
    
        with trange(CFG['episodes_per_epoch'], desc=f"Epoch {epoch+1}/{CFG['epochs']}") as t:
            for episode_idx in t:
                example = None
                reference_makespan = None
                if CFG['use_random_problems']:
                    instance = generate_random_instance(
                        num_jobs=CFG['random_jobs'],
                        num_machines=CFG['random_machines'],
                        process_time_range=(CFG['random_time_low'], CFG['random_time_high']),
                        rng=rng,
                    )
                    inst = instance['inst_for_ortools']
                    problem_jobs = int(len(inst))
                    problem_machines = int(len(inst[0]) if inst else 0)
                else:
                    example = random.choice(dataset_split)
                    inst = parse_prompt_jobs_first(example['prompt_jobs_first'])
                    problem_jobs = int(example.get('num_jobs', len(inst)))
                    problem_machines = int(example.get('num_machines', len(inst[0]) if inst else 0))
                    reference_makespan = _extract_reference_makespan(example)
    
                heuristic_makespan = None
                if CFG['reward_mode'] == 'mwkr_relative':
                    _, heuristic_makespan = mwkr_schedule(inst)
    
                if CFG['rl_algo'] in {'grpo_episode', 'grpo_manual'}:
                    group_rollouts = []
                    rewards = []
                    for _ in range(max(1, CFG['group_size'])):
                        rollout_cfg = dict(CFG)
                        rollout_cfg['action_code_seed'] = int(rng.integers(0, 2**31 - 1))
                        try:
                            makespan, feasible, traces = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
                        except StepActionParseError as exc:
                            makespan = float(CFG['invalid_makespan_penalty'])
                            feasible = False
                            traces = []
                            if CFG.get('print_step_trace', False):
                                print(f'[warn] GRPO rollout parse failed -> penalty applied: {exc}')
    
                        reward = compute_episode_reward(
                            makespan=makespan,
                            feasible=bool(feasible),
                            inst_for_ortools=inst,
                            invalid_makespan_penalty=float(CFG['invalid_makespan_penalty']),
                            reward_mode=CFG['reward_mode'],
                            heuristic_makespan=heuristic_makespan,
                        )
    
                        step_samples = []
                        for tr in traces:
                            old_lp = compute_log_prob_mean(
                                model=model,
                                sequence_ids=tr.sequence_ids,
                                prompt_len=tr.prompt_len,
                                action_token_id=tr.action_token_id,
                                device=device,
                                require_grad=False,
                            ).detach().cpu()
                            step_samples.append({
                                'sequence_ids': tr.sequence_ids.detach().cpu(),
                                'prompt_len': tr.prompt_len,
                                'action_token_id': int(tr.action_token_id),
                                'old_log_prob': old_lp,
                            })
    
                        group_rollouts.append(
                            {
                                'reward': float(reward),
                                'feasible': bool(feasible),
                                'makespan': float(makespan),
                                'step_samples': step_samples,
                            }
                        )
                        rewards.append(float(reward))
    
                    rewards_t = torch.tensor(rewards, dtype=torch.float32)
                    mean_r = float(rewards_t.mean().item())
                    std_r = float(rewards_t.std(unbiased=False).item())
                    denom = std_r + 1e-8
    
                    for rollout in group_rollouts:
                        advantage = (rollout['reward'] - mean_r) / denom
                        for s in rollout['step_samples']:
                            grpo_samples.append(
                                TrajectorySample(
                                    sequence_ids=s['sequence_ids'],
                                    prompt_len=s['prompt_len'],
                                    action_token_id=int(s['action_token_id']),
                                    reward=rollout['reward'],
                                    advantage=float(advantage),
                                    old_log_prob=s['old_log_prob'],
                                    feasible=rollout['feasible'],
                                    makespan=rollout['makespan'],
                                )
                            )
    
                    feasible_makespans = [r['makespan'] for r in group_rollouts if r['feasible']]
                    feasible_rollout_count = len(feasible_makespans)
                    best_ms = min(feasible_makespans) if feasible_makespans else float(CFG['invalid_makespan_penalty'])
                    ref_gap = None
                    if reference_makespan not in (None, 0) and feasible_rollout_count > 0:
                        ref_gap = (float(best_ms) - float(reference_makespan)) / float(reference_makespan)
    
                    epoch_best_makespans.append(float(best_ms))
                    epoch_feasible_rollout_counts.append(int(feasible_rollout_count))
                    epoch_group_rewards.append(float(mean_r))
                    if ref_gap is not None:
                        epoch_reference_gaps.append(float(ref_gap))
    
                    episode_history.append({
                        'epoch': int(epoch + 1),
                        'episode': int(episode_idx + 1),
                        'algo': CFG['rl_algo'],
                        'num_jobs': int(problem_jobs),
                        'num_machines': int(problem_machines),
                        'reference_makespan': reference_makespan,
                        'best_makespan': float(best_ms),
                        'gap_vs_reference': ref_gap,
                        'feasible_rollouts': int(feasible_rollout_count),
                        'group_size': int(max(1, CFG['group_size'])),
                        'group_mean_reward': float(mean_r),
                        'pair_count': None,
                        'status': 'ok',
                    })
    
                    t.set_postfix(
                        algo='grpo-episode',
                        opt_ms=(f'{reference_makespan:.1f}' if reference_makespan is not None else 'NA'),
                        best_ms=f'{best_ms:.1f}',
                        gap_pct=_format_gap_pct(ref_gap),
                        reward=f'{mean_r:.1f}',
                    )
                    if CFG.get('print_episode_summary', True):
                        _print_episode_result(
                            epoch,
                            episode_idx,
                            CFG['episodes_per_epoch'],
                            algo='grpo_episode',
                            num_jobs=problem_jobs,
                            num_machines=problem_machines,
                            reference_makespan=reference_makespan,
                            best_makespan=best_ms,
                            gap_vs_reference=ref_gap,
                            pair_count=None,
                            feasible_rollouts=feasible_rollout_count,
                            note=f'group_reward={mean_r:.1f}',
                        )
    
                elif CFG['rl_algo'] == 'bopo':
                    group_rollouts = []
                    for _ in range(max(2, int(CFG['group_size']))):
                        rollout_cfg = dict(CFG)
                        rollout_cfg['action_code_seed'] = int(rng.integers(0, 2**31 - 1))
                        try:
                            makespan, feasible, traces = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
                        except StepActionParseError as exc:
                            makespan = float(CFG['invalid_makespan_penalty'])
                            feasible = False
                            traces = []
                            if CFG.get('print_step_trace', False):
                                print(f'[warn] BOPO rollout parse failed -> penalty applied: {exc}')
    
                        group_rollouts.append(
                            {
                                'feasible': bool(feasible),
                                'makespan': float(makespan),
                                'traces': traces,
                            }
                        )
    
                    feasible_makespans = [r['makespan'] for r in group_rollouts if r['feasible']]
                    feasible_rollout_count = len(feasible_makespans)
                    best_ms = min(feasible_makespans) if feasible_makespans else float(CFG['invalid_makespan_penalty'])
    
                    bopo_pairs = build_bopo_step_pairs(
                        group_rollouts=group_rollouts,
                        rng=rng,
                        min_relative_gap=CFG['bopo_min_relative_gap'],
                        max_pairs_per_group=CFG['bopo_max_pairs_per_group'],
                        max_step_pairs_per_pair=CFG['bopo_max_step_pairs_per_pair'],
                        pair_mode=CFG['bopo_pair_mode'],
                    )
    
                    ref_gap = None
                    if reference_makespan not in (None, 0) and feasible_rollout_count > 0:
                        ref_gap = (float(best_ms) - float(reference_makespan)) / float(reference_makespan)
    
                    epoch_best_makespans.append(float(best_ms))
                    epoch_feasible_rollout_counts.append(int(feasible_rollout_count))
                    epoch_pair_counts.append(int(len(bopo_pairs)))
                    if ref_gap is not None:
                        epoch_reference_gaps.append(float(ref_gap))
    
                    episode_history.append({
                        'epoch': int(epoch + 1),
                        'episode': int(episode_idx + 1),
                        'algo': CFG['rl_algo'],
                        'num_jobs': int(problem_jobs),
                        'num_machines': int(problem_machines),
                        'reference_makespan': reference_makespan,
                        'best_makespan': float(best_ms),
                        'gap_vs_reference': ref_gap,
                        'feasible_rollouts': int(feasible_rollout_count),
                        'group_size': int(max(2, CFG['group_size'])),
                        'group_mean_reward': None,
                        'pair_count': int(len(bopo_pairs)),
                        'status': 'ok' if len(bopo_pairs) > 0 else 'no_pairs',
                    })
    
                    if not bopo_pairs:
                        t.set_postfix(
                            algo='bopo-step',
                            opt_ms=(f'{reference_makespan:.1f}' if reference_makespan is not None else 'NA'),
                            best_ms=f'{best_ms:.1f}',
                            gap_pct=_format_gap_pct(ref_gap),
                            pairs=0,
                        )
                        if CFG.get('print_episode_summary', True):
                            _print_episode_result(
                                epoch,
                                episode_idx,
                                CFG['episodes_per_epoch'],
                                algo='bopo',
                                num_jobs=problem_jobs,
                                num_machines=problem_machines,
                                reference_makespan=reference_makespan,
                                best_makespan=best_ms,
                                gap_vs_reference=ref_gap,
                                pair_count=0,
                                feasible_rollouts=feasible_rollout_count,
                                note='no_pairs',
                            )
                        continue
    
                    bopo_pairs_epoch.extend(bopo_pairs)
    
                    t.set_postfix(
                        algo='bopo-collect',
                        opt_ms=(f'{reference_makespan:.1f}' if reference_makespan is not None else 'NA'),
                        best_ms=f'{best_ms:.1f}',
                        gap_pct=_format_gap_pct(ref_gap),
                        pairs=len(bopo_pairs),
                        epoch_pairs=len(bopo_pairs_epoch),
                    )
                    if CFG.get('print_episode_summary', True):
                        _print_episode_result(
                            epoch,
                            episode_idx,
                            CFG['episodes_per_epoch'],
                            algo='bopo',
                            num_jobs=problem_jobs,
                            num_machines=problem_machines,
                            reference_makespan=reference_makespan,
                            best_makespan=best_ms,
                            gap_vs_reference=ref_gap,
                            pair_count=len(bopo_pairs),
                            feasible_rollouts=feasible_rollout_count,
                            note=f'epoch_pairs={len(bopo_pairs_epoch)}',
                        )
    
                    log_every = max(1, int(CFG.get('rl_log_every_episodes', 5)))
                    if ((episode_idx + 1) % log_every) == 0:
                        _print_running_epoch_status(
                            epoch,
                            episode_idx,
                            CFG,
                            best_makespans=epoch_best_makespans,
                            gaps=epoch_reference_gaps,
                            pair_counts=epoch_pair_counts,
                            feasible_counts=epoch_feasible_rollout_counts,
                            note=f'bopo_collect epoch_pairs={len(bopo_pairs_epoch)}',
                        )
    
                    update_every = int(CFG.get('bopo_update_every_episodes', 0) or 0)
                    if update_every > 0 and ((episode_idx + 1) % update_every) == 0 and bopo_pairs_epoch:
                        bopo_loss, bopo_gap, pair_updates = bopo_step(
                            pairs=bopo_pairs_epoch,
                            model=model,
                            optimizer=optimizer,
                            device=device,
                            beta=CFG['bopo_beta'],
                            gap_scale=CFG['bopo_gap_scale'],
                            margin=CFG['bopo_margin'],
                        )
                        bopo_updates += int(pair_updates)
                        bopo_loss_total += float(bopo_loss) * max(int(pair_updates), 1)
                        bopo_gap_total += float(bopo_gap) * max(int(pair_updates), 1)
                        avg_pair_loss_running = float(bopo_loss_total) / float(max(bopo_updates, 1))
                        avg_gap_running = float(bopo_gap_total) / float(max(bopo_updates, 1))
                        print(
                            f"[Epoch {epoch+1} Episode {episode_idx+1}] BOPO pair-updates={pair_updates}, "
                            f"avg_pair_loss={avg_pair_loss_running:.6f}, avg_rel_gap={avg_gap_running:.6f}, buffered_pairs={len(bopo_pairs_epoch)}"
                        )
                        bopo_pairs_epoch = []
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
    
                else:
                    rollout_cfg = dict(CFG)
                    rollout_cfg['action_code_seed'] = int(rng.integers(0, 2**31 - 1))
                    try:
                        makespan, feasible, traces = rollout_step_episode(model, tokenizer, inst, device, rollout_cfg)
                    except StepActionParseError as exc:
                        if CFG.get('print_step_trace', False):
                            print(f'[warn] REINFORCE rollout parse failed -> skip: {exc}')
                        episode_history.append({
                            'epoch': int(epoch + 1),
                            'episode': int(episode_idx + 1),
                            'algo': CFG['rl_algo'],
                            'num_jobs': int(problem_jobs),
                            'num_machines': int(problem_machines),
                            'reference_makespan': reference_makespan,
                            'best_makespan': None,
                            'gap_vs_reference': None,
                            'feasible_rollouts': 0,
                            'group_size': 1,
                            'group_mean_reward': None,
                            'pair_count': None,
                            'status': 'parse_failed',
                        })
                        t.set_postfix_str('parse-failed rollout, skipping')
                        continue
                    if not feasible or not traces or not math.isfinite(makespan):
                        episode_history.append({
                            'epoch': int(epoch + 1),
                            'episode': int(episode_idx + 1),
                            'algo': CFG['rl_algo'],
                            'num_jobs': int(problem_jobs),
                            'num_machines': int(problem_machines),
                            'reference_makespan': reference_makespan,
                            'best_makespan': None,
                            'gap_vs_reference': None,
                            'feasible_rollouts': 0,
                            'group_size': 1,
                            'group_mean_reward': None,
                            'pair_count': None,
                            'status': 'invalid',
                        })
                        t.set_postfix_str('invalid rollout, skipping')
                        continue
    
                    reward = compute_episode_reward(
                        makespan=makespan,
                        feasible=bool(feasible),
                        inst_for_ortools=inst,
                        invalid_makespan_penalty=float(CFG['invalid_makespan_penalty']),
                        reward_mode=CFG['reward_mode'],
                        heuristic_makespan=heuristic_makespan,
                    )
                    mwkr_makespan = (
                        float(heuristic_makespan)
                        if heuristic_makespan is not None
                        else float(mwkr_schedule(inst)[1])
                    )
                    baseline = compute_episode_reward(
                        makespan=mwkr_makespan,
                        feasible=math.isfinite(mwkr_makespan),
                        inst_for_ortools=inst,
                        invalid_makespan_penalty=float(CFG['invalid_makespan_penalty']),
                        reward_mode=CFG['reward_mode'],
                        heuristic_makespan=mwkr_makespan,
                    )
                    if CFG['use_running_baseline']:
                        baseline = baseline_tracker.update(reward)
                    advantage = reward - baseline
    
                    adv_tensor = torch.tensor(float(advantage), device=device)
                    trace_update_count = 0
                    trace_loss_sum = 0.0
                    for tr in traces:
                        lp = compute_log_prob_mean(
                            model=model,
                            sequence_ids=tr.sequence_ids,
                            prompt_len=tr.prompt_len,
                            action_token_id=tr.action_token_id,
                            device=device,
                            require_grad=True,
                        )
                        loss = -(lp * adv_tensor)
                        optimizer.zero_grad()
                        loss.backward()
                        params = []
                        for g in optimizer.param_groups:
                            params.extend(g['params'])
                        torch.nn.utils.clip_grad_norm_(params, 1.0)
                        optimizer.step()
    
                        trace_update_count += 1
                        trace_loss_sum += float(loss.item())
                        del lp, loss
    
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
    
                    reinforce_updates += int(trace_update_count)
                    reinforce_loss_sum += float(trace_loss_sum)
                    avg_trace_loss = (trace_loss_sum / trace_update_count) if trace_update_count > 0 else 0.0
                    ref_gap = None
                    if reference_makespan not in (None, 0):
                        ref_gap = (float(makespan) - float(reference_makespan)) / float(reference_makespan)
    
                    epoch_best_makespans.append(float(makespan))
                    epoch_feasible_rollout_counts.append(1)
                    epoch_advantages.append(float(advantage))
                    epoch_trace_losses.append(float(avg_trace_loss))
                    if ref_gap is not None:
                        epoch_reference_gaps.append(float(ref_gap))
    
                    episode_history.append({
                        'epoch': int(epoch + 1),
                        'episode': int(episode_idx + 1),
                        'algo': CFG['rl_algo'],
                        'num_jobs': int(problem_jobs),
                        'num_machines': int(problem_machines),
                        'reference_makespan': reference_makespan,
                        'best_makespan': float(makespan),
                        'gap_vs_reference': ref_gap,
                        'feasible_rollouts': 1,
                        'group_size': 1,
                        'group_mean_reward': float(reward),
                        'pair_count': None,
                        'status': 'ok',
                    })
    
                    t.set_postfix(
                        algo='reinforce-step',
                        opt_ms=(f'{reference_makespan:.1f}' if reference_makespan is not None else 'NA'),
                        best_ms=f'{makespan:.1f}',
                        gap_pct=_format_gap_pct(ref_gap),
                        reward=f'{reward:.1f}',
                        advantage=f'{advantage:.1f}',
                        steps=len(traces),
                        trace_updates=trace_update_count,
                        loss=f'{avg_trace_loss:.4f}',
                    )
                    if CFG.get('print_episode_summary', True):
                        _print_episode_result(
                            epoch,
                            episode_idx,
                            CFG['episodes_per_epoch'],
                            algo='reinforce',
                            num_jobs=problem_jobs,
                            num_machines=problem_machines,
                            reference_makespan=reference_makespan,
                            best_makespan=makespan,
                            gap_vs_reference=ref_gap,
                            pair_count=None,
                            feasible_rollouts=1,
                            note=f'reward={reward:.1f}, loss={avg_trace_loss:.4f}',
                        )
    
        epoch_row = {
            'epoch': int(epoch + 1),
            'algo': CFG['rl_algo'],
            'episodes': int(CFG['episodes_per_epoch']),
            'group_size': int(max(1, CFG['group_size'])) if CFG['rl_algo'] in {'grpo_episode', 'grpo_manual', 'bopo'} else 1,
            'avg_best_makespan': _safe_mean(epoch_best_makespans),
            'min_best_makespan': min(epoch_best_makespans) if epoch_best_makespans else None,
            'avg_gap_vs_reference': _safe_mean(epoch_reference_gaps),
            'median_gap_vs_reference': _safe_median(epoch_reference_gaps),
            'reference_episode_count': int(len(epoch_reference_gaps)),
            'avg_feasible_rollouts': _safe_mean(epoch_feasible_rollout_counts),
            'avg_pair_count': _safe_mean(epoch_pair_counts),
            'avg_group_reward': _safe_mean(epoch_group_rewards),
            'avg_advantage': _safe_mean(epoch_advantages),
            'avg_trace_loss': _safe_mean(epoch_trace_losses),
            'pair_updates': 0,
            'avg_pair_loss': None,
            'avg_rel_gap': None,
            'grpo_samples': 0,
            'grpo_loss': None,
            'grpo_approx_kl': None,
            'reinforce_trace_updates': int(reinforce_updates),
        }
    
        if CFG['rl_algo'] in {'grpo_episode', 'grpo_manual'}:
            if not grpo_samples:
                print(f'[Epoch {epoch+1}] no GRPO samples; skipping update')
            else:
                loss_value, approx_kl = grpo_step(
                    samples=grpo_samples,
                    model=model,
                    optimizer=optimizer,
                    device=device,
                    clip_epsilon=CFG['clip_epsilon'],
                    grpo_epochs=CFG['grpo_epochs'],
                    kl_coef=CFG['kl_coef'],
                    update_batch_size=CFG['grpo_update_batch_size'],
                )
                epoch_row['grpo_samples'] = int(len(grpo_samples))
                epoch_row['grpo_loss'] = float(loss_value)
                epoch_row['grpo_approx_kl'] = float(approx_kl)
                print(f"[Epoch {epoch+1}] GRPO loss={loss_value:.6f}, approx_kl={approx_kl:.6f}, samples={len(grpo_samples)}")
        elif CFG['rl_algo'] == 'bopo':
            if bopo_pairs_epoch:
                bopo_loss, bopo_gap, pair_updates = bopo_step(
                    pairs=bopo_pairs_epoch,
                    model=model,
                    optimizer=optimizer,
                    device=device,
                    beta=CFG['bopo_beta'],
                    gap_scale=CFG['bopo_gap_scale'],
                    margin=CFG['bopo_margin'],
                )
                bopo_updates += int(pair_updates)
                bopo_loss_total += float(bopo_loss) * max(int(pair_updates), 1)
                bopo_gap_total += float(bopo_gap) * max(int(pair_updates), 1)
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    
            if bopo_updates <= 0:
                print(f'[Epoch {epoch+1}] no valid BOPO preference pairs; skipping update')
            else:
                avg_pair_loss = float(bopo_loss_total) / float(max(bopo_updates, 1))
                avg_gap = float(bopo_gap_total) / float(max(bopo_updates, 1))
                epoch_row['pair_updates'] = int(bopo_updates)
                epoch_row['avg_pair_loss'] = float(avg_pair_loss)
                epoch_row['avg_rel_gap'] = float(avg_gap)
                print(
                    f"[Epoch {epoch+1}] BOPO pair-updates={bopo_updates}, "
                    f"avg_pair_loss={avg_pair_loss:.6f}, avg_rel_gap={avg_gap:.6f}"
                )
        else:
            if reinforce_updates <= 0:
                print(f'[Epoch {epoch+1}] no valid episodes; skipping update')
            else:
                avg_loss = reinforce_loss_sum / max(reinforce_updates, 1)
                epoch_row['avg_trace_loss'] = float(avg_loss)
                print(
                    f"[Epoch {epoch+1}] REINFORCE trace-updates={reinforce_updates}, "
                    f"avg_trace_loss={avg_loss:.6f}"
                )
    
        epoch_history.append(epoch_row)
        if CFG.get('save_rl_epoch_metrics', True):
            _write_history_csv(epoch_metrics_path, epoch_history)
        if CFG.get('save_rl_episode_metrics', True):
            _write_history_csv(episode_metrics_path, episode_history)
        print('epoch metrics saved:', epoch_metrics_path)
        print('episode metrics saved:', episode_metrics_path)
    
        save_interval = max(1, int(CFG.get('save_every_n_epochs', 1) or 1))
        if CFG['save_every_epoch'] and (((epoch + 1) % save_interval) == 0):
            ckpt = output_dir / f'checkpoint-epoch-{epoch+1}'
            ckpt.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(str(ckpt))
            tokenizer.save_pretrained(str(ckpt))
            print('saved:', ckpt)
    
    final_dir = output_dir / 'final'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    
    print('training done')
    print('output_dir:', output_dir)
    print('final_dir:', final_dir)
    print('epoch_metrics_path:', epoch_metrics_path)
    print('episode_metrics_path:', episode_metrics_path)
    print('checkpoints:')
    for p in sorted(output_dir.glob('checkpoint-epoch-*')):
        print(' -', p)


## 4) RL metrics summary / plot


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path(CFG['output_dir'])
epoch_metrics_path = output_dir / 'rl_epoch_metrics.csv'
episode_metrics_path = output_dir / 'rl_episode_metrics.csv'
grpo_log_history_path = output_dir / 'grpo_log_history.csv'
print('epoch_metrics_path:', epoch_metrics_path)
print('episode_metrics_path:', episode_metrics_path)
print('grpo_log_history_path:', grpo_log_history_path)

if epoch_metrics_path.exists():
    epoch_df = pd.read_csv(epoch_metrics_path)
    display(epoch_df.tail(20))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.flatten()

    axes[0].plot(epoch_df['epoch'], epoch_df['avg_best_makespan'], marker='o')
    axes[0].set_title('Avg Best Makespan')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Makespan')

    if 'avg_gap_vs_reference' in epoch_df.columns and epoch_df['avg_gap_vs_reference'].notna().any():
        axes[1].plot(epoch_df['epoch'], epoch_df['avg_gap_vs_reference'], marker='o', color='tab:orange')
        axes[1].set_title('Avg Gap Vs Reference')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Gap')
    else:
        axes[1].axis('off')

    if 'avg_pair_loss' in epoch_df.columns and epoch_df['avg_pair_loss'].notna().any():
        axes[2].plot(epoch_df['epoch'], epoch_df['avg_pair_loss'], marker='o', color='tab:green')
        axes[2].set_title('BOPO Avg Pair Loss')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Loss')
    elif 'avg_trace_loss' in epoch_df.columns and epoch_df['avg_trace_loss'].notna().any():
        axes[2].plot(epoch_df['epoch'], epoch_df['avg_trace_loss'], marker='o', color='tab:green')
        axes[2].set_title('Avg Trace Loss')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Loss')
    else:
        axes[2].axis('off')

    if 'avg_rel_gap' in epoch_df.columns and epoch_df['avg_rel_gap'].notna().any():
        axes[3].plot(epoch_df['epoch'], epoch_df['avg_rel_gap'], marker='o', color='tab:red')
        axes[3].set_title('BOPO Avg Relative Gap')
        axes[3].set_xlabel('Epoch')
        axes[3].set_ylabel('Gap')
    else:
        axes[3].axis('off')

    plt.tight_layout()
    plt.show()

    if episode_metrics_path.exists():
        episode_df = pd.read_csv(episode_metrics_path)
        display(episode_df.tail(20))
elif grpo_log_history_path.exists():
    grpo_df = pd.read_csv(grpo_log_history_path)
    display(grpo_df.tail(30))
    numeric_cols = [c for c in grpo_df.columns if pd.api.types.is_numeric_dtype(grpo_df[c])]
    reward_cols = [c for c in numeric_cols if 'reward' in c.lower()]
    loss_cols = [c for c in numeric_cols if 'loss' in c.lower() or 'kl' in c.lower()]
    step_col = 'step' if 'step' in grpo_df.columns else None
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    if step_col and reward_cols:
        for col in reward_cols[:4]:
            axes[0].plot(grpo_df[step_col], grpo_df[col], marker='o', label=col)
        axes[0].set_title('GRPO Rewards')
        axes[0].set_xlabel('Step')
        axes[0].legend()
    else:
        axes[0].axis('off')
    if step_col and loss_cols:
        for col in loss_cols[:4]:
            axes[1].plot(grpo_df[step_col], grpo_df[col], marker='o', label=col)
        axes[1].set_title('GRPO Loss / KL')
        axes[1].set_xlabel('Step')
        axes[1].legend()
    else:
        axes[1].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No RL metrics file found. Run the training cell first.')


## 4) (선택) RL 모델 HF 업로드


In [ ]:
if CFG['upload_after_train']:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=CFG['hf_model_repo_out'], repo_type='model', private=False, exist_ok=True)

    output_dir = Path(CFG['output_dir'])
    upload_dir = resolve_upload_dir(output_dir, CFG['upload_source'], CFG['checkpoint_tag'])
    print('upload_dir:', upload_dir)

    upload_folder(
        repo_id=CFG['hf_model_repo_out'],
        repo_type='model',
        folder_path=str(upload_dir),
        token=HF_TOKEN,
        commit_message=f"Upload RL adapter ({upload_dir.name}) from colab_05_full",
    )

    files = api.list_repo_files(repo_id=CFG['hf_model_repo_out'], repo_type='model')
    print(f"upload done: {CFG['hf_model_repo_out']} ({len(files)} files)")
    for f in files:
        print(' -', f)
else:
    print('CFG[upload_after_train]=False, skip upload')
